## Case 1 — Cryptocurrency Price Movement Prediction

This notebook builds BTC and ETH price movement prediction pipelines from multiple data sources:
- CoinGecko daily crypto data
- Google Trends weekly search data
- Effective Federal Funds Rate (EFFR)
- Traditional market proxies: QQQ (Nasdaq-100 ETF proxy), UUP (US dollar index ETF proxy), GLD (gold ETF proxy), BSV (bond / fixed-income ETF proxy)

Main workflow:
1. Load raw CSV files
2. Align everything to a daily index
3. Build technical / lagged / regime / interaction features
4. Split data into train / validation / test
5. Train and tune multiple classifiers
6. Compare model performance and inspect feature importance
7. Select the best model by validation and demonstrate final predictions on held-out test data

## Section 1: Data Loading & Standardization

### 1. What This Code Does

This code implements the **data loading layer** of the prediction pipeline.

Its purpose is to:
- Load raw datasets from multiple sources
- Standardize their formats (especially time columns)
- Store them in a unified structure for later use

It does **not perform modeling**, but prepares clean inputs for later steps.

---

### 2. Why This Step Is Necessary

The data comes from different sources with:
- Different formats (daily vs weekly)
- Different column names
- Different levels of data quality

Without standardization:
- Datasets cannot be merged correctly
- Time-series order may be wrong → leads to data leakage
- Feature engineering (e.g., lag variables) becomes invalid

---

### 3. What Data Is Loaded

Three types of data are included:

#### (1) Crypto Market Data
- Bitcoin and Ethereum daily data  
- Core signals for prediction  

#### (2) Traditional & Macro Data
- Equity (QQQ), Dollar (UUP), Gold (GLD), Bonds (BSV), Interest Rate (EFFR)  
- Capture external financial environment  

#### (3) Google Trends Data
- Search interest for crypto-related keywords  
- Proxy for investor attention and sentiment  

---

### 4. How the Code Works

#### (1) Central File Registry
All file paths are defined in one place to improve maintainability and reproducibility.

#### (2) Modular Data Readers
Different functions handle different data formats:
- Crypto data
- Market data
- Google Trends (special format)
- Macro data

This avoids errors caused by inconsistent structures.

#### (3) Core Standardization Steps

All datasets go through:

- **Datetime parsing** → ensures consistent time format  
- **Chronological sorting** → required for time-series modeling  
- **Index reset** → keeps data structure clean  

#### (4) Unified Storage

All datasets are stored in a single dictionary:
- Easy to access
- Easy to merge later
- Supports scalable pipeline design

---

### 5. Summary

This code builds a **clean, standardized, and unified data layer**, ensuring that:

- All datasets are time-consistent  
- Data is ready for merging and feature construction  
- The prediction pipeline can run reliably  

In [17]:
# Cell 1 — Load the raw CSV files
#
# This cell only reads and standardizes the raw inputs.
# The output is a `datasets` dictionary containing pandas DataFrames keyed by source name.
# Dates are parsed immediately because every later merge depends on consistent datetime columns.
# Google Trends files contain metadata rows above the real header, so they need `skiprows=2`.

from __future__ import annotations  # allow modern type hints without string quoting in older Python versions

from pathlib import Path  # convenient cross-platform path handling
import warnings  # used to suppress noisy but non-fatal runtime warnings
import numpy as np  # numerical operations
import pandas as pd  # tabular data processing
from sklearn.exceptions import ConvergenceWarning  # warning class emitted by some sklearn models

pd.set_option("display.max_columns", 200)  # show many columns when previewing wide feature tables
pd.set_option("display.width", 140)  # widen console-style DataFrame rendering for readability

# Suppress sklearn convergence warnings so notebook output stays focused on results, not optimizer noise.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
# Suppress repeated numeric warnings from sklearn matrix multiplications after sanitization/clipping steps.
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"sklearn\.utils\.extmath",
)

# Tell numpy not to spam warnings for overflow/divide-by-zero in intermediate calculations.
np.seterr(over="ignore", divide="ignore", invalid="ignore")

# Absolute root directory for all files used in this notebook.
BASE_DIR = Path(".")

# Central file registry so file locations are defined once and reused everywhere.
FILES = {
    "eth_daily": BASE_DIR / "ethereum_daily.csv",  # Ethereum CoinGecko daily data
    "btc_daily": BASE_DIR / "bitcoin_daily.csv",  # Bitcoin CoinGecko daily data
    "effr": BASE_DIR / "Effective Federal Funds Rate.csv",  # effective federal funds rate
    "qqq": BASE_DIR / "QQQ_daily_2025-01-10_2026-01-15.csv",  # Nasdaq-100 ETF proxy
    "uup": BASE_DIR / "UUP_daily_2025-01-10_2026-01-15.csv",  # US dollar index ETF proxy
    "gld": BASE_DIR / "GLD_daily_2025-01-10_2026-01-15.csv",  # gold ETF proxy
    "bsv": BASE_DIR / "BSV_daily_2025-01-10_2026-01-15.csv",  # bond / fixed-income proxy file provided by user
    "gt_bitcoin": BASE_DIR / "Bitcoin_GT.csv",  # weekly Google Trends for Bitcoin
    "gt_ethereum": BASE_DIR / "Ethereum_GT.csv",  # weekly Google Trends for Ethereum
    "gt_crypto": BASE_DIR / "CryptoCurrency.csv",  # weekly Google Trends for Cryptocurrency keyword
    "gt_btc_pm": BASE_DIR / "BTC_PM.csv",  # weekly Google Trends for bitcoin price manipulation
    "gt_eth_pm": BASE_DIR / "ETH_PM.csv",  # weekly Google Trends for ethereum price manipulation
}


# Read a daily OHLCV-like market file and standardize its date column.
def read_ohlcv_daily(path: Path, date_col: str = "Date") -> pd.DataFrame:
    df = pd.read_csv(path)  # load the CSV into pandas
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")  # convert date text to datetime; invalid values become NaT
    df = df.sort_values(date_col).reset_index(drop=True)  # sort in chronological order and reset row index
    return df  # return the cleaned DataFrame


# Read a CoinGecko crypto daily file and standardize the built-in `date` column.
def read_coingecko_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)  # load the CSV into pandas
    df["date"] = pd.to_datetime(df["date"], errors="coerce")  # parse CoinGecko date column
    df = df.sort_values("date").reset_index(drop=True)  # ensure chronological order
    return df  # return the cleaned DataFrame


# Read a Google Trends weekly file that includes two metadata rows above the real header.
def read_google_trends_weekly(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=2)  # skip metadata so the third row becomes the header
    df.columns = [c.strip() for c in df.columns]  # remove extra whitespace from column names
    if "Week" in df.columns:  # only process if the expected time column exists
        df["Week"] = pd.to_datetime(df["Week"], errors="coerce")  # parse weekly timestamp
        df = df.sort_values("Week").reset_index(drop=True)  # keep observations ordered by week
    return df  # return standardized weekly trends DataFrame


# Read the effective federal funds rate file and standardize its date column.
def read_effr(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)  # load the CSV
    df.columns = [c.strip() for c in df.columns]  # strip whitespace from headers just in case
    if "observation_date" in df.columns:  # verify expected date column is present
        df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")  # parse dates
        df = df.sort_values("observation_date").reset_index(drop=True)  # sort in time order
    return df  # return standardized macro DataFrame


# Container for every raw dataset after loading and light standardization.
datasets: dict[str, pd.DataFrame] = {}

# Load BTC and ETH raw crypto datasets.
datasets["btc_daily"] = read_coingecko_daily(FILES["btc_daily"])
datasets["eth_daily"] = read_coingecko_daily(FILES["eth_daily"])

# Load traditional market proxy datasets.
datasets["qqq"] = read_ohlcv_daily(FILES["qqq"])
datasets["uup"] = read_ohlcv_daily(FILES["uup"])
datasets["gld"] = read_ohlcv_daily(FILES["gld"])
datasets["bsv"] = read_ohlcv_daily(FILES["bsv"])

# Load macro dataset.
datasets["effr"] = read_effr(FILES["effr"])

# Load weekly Google Trends datasets.
datasets["gt_bitcoin"] = read_google_trends_weekly(FILES["gt_bitcoin"])
datasets["gt_ethereum"] = read_google_trends_weekly(FILES["gt_ethereum"])
datasets["gt_crypto"] = read_google_trends_weekly(FILES["gt_crypto"])
datasets["gt_btc_pm"] = read_google_trends_weekly(FILES["gt_btc_pm"])
datasets["gt_eth_pm"] = read_google_trends_weekly(FILES["gt_eth_pm"])

{k: v.shape for k, v in datasets.items()}  # quick check: number of rows/columns loaded for each source

{'btc_daily': (366, 8),
 'eth_daily': (366, 8),
 'qqq': (255, 6),
 'uup': (255, 6),
 'gld': (255, 6),
 'bsv': (255, 6),
 'effr': (260, 2),
 'gt_bitcoin': (53, 2),
 'gt_ethereum': (53, 2),
 'gt_crypto': (53, 2),
 'gt_btc_pm': (53, 2),
 'gt_eth_pm': (53, 2)}

## Section 2 — Feature Engineering

### 1. What This Section Does

This section builds the **core modeling dataset** by transforming raw data into structured features.

It performs four key tasks:

1. Align all datasets to a **common daily timeline**  
2. Generate **technical, lagged, regime, and interaction features**  
3. Integrate **macro, cross-asset, and behavioral data**  
4. Construct the **final input features (X) and prediction target (y)**  

👉 This is where **raw data → predictive signals**

---

### 2. Dataset Classification

All input tables are grouped into three categories:

---

#### 🔵 2.1 Crypto Market Tables (Core Layer)

**Tables:**
- `btc_daily`
- `eth_daily`

**Feature Categories:**

##### (1) Raw Features (Original)
- close price  
- trading volume  

👉 Base variables used to derive almost all features  

---

##### (2) Return Features (Derived)
- 1-day return  
- 3-day / 7-day / 30-day return  
- log return  

👉 Capture price dynamics and short-term momentum  

---

##### (3) Lagged Features (Derived)
- return_lag1, lag2, lag3  
- volume_lag1  
- RSI_lag1  

👉 Capture **autocorrelation (time dependency)**  

---

##### (4) Volatility Features (Derived)
- rolling std (7d, 14d, 30d)  

👉 Measure risk / uncertainty  

---

##### (5) Trend Features (Derived)
- moving averages (MA)  
- MA ratios  

👉 Capture medium-term direction  

---

##### (6) Momentum / Oscillators (Derived)
- RSI (Relative Strength Index)  
- MACD (Moving Average Convergence and Divergence) / signal / histogram  

👉 Capture overbought / oversold conditions  

---

##### (7) Volume Features (Derived)
- volume change  
- volume MA  
- volume ratio  

👉 Reflect liquidity and trading intensity  

---

##### (8) Regime Features (Derived)
- bull_market_dummy (30d return > 0)  
- volatility_regime  

👉 Capture market states  

---

##### (9) Interaction Features (Derived)
- return × volume  
- return × regime  

👉 Capture nonlinear effects  

---

---

#### 🟢 2.2 Traditional Market & Macro Tables (Context Layer)

**Tables:**
- `qqq` (equity market)
- `uup` (USD index)
- `gld` (gold)
- `bsv` (bonds)
- `effective federal fund rate (effr)` (interest rate)

**Feature Categories:**

##### (1) Raw Features
- close price of each asset  
- interest rate level  

---

##### (2) Derived Features
- daily returns (e.g., QQQ return)  
- rolling volatility (QQQ)  

---

##### (3) Regime Features (Derived)
- VIX_high_regime (equity volatility proxy)  
- DXY_trend_regime (USD trend)  

👉 Represent macro conditions:
- risk-on vs risk-off  
- liquidity environment  

---

##### (4) Interaction Features (Derived)
- crypto return × QQQ return  
- crypto return × VIX regime  
- crypto return × gold return  

👉 Capture cross-asset influence  

---

---

#### 🔴 2.3 Google Trends Tables (Behavior Layer)

**Tables:**
- asset-specific search (BTC / ETH) `Bitcoin_GT` `Ethereum_GT`
- manipulation-related search `Bitcoin_PM` `Ethereum_PM`  
- general crypto search `Cryptocurrency`

**Feature Categories:**

##### (1) Raw Features
- search intensity values  

👉 Proxy for investor attention  

---

##### (2) Processed Features
- converted from weekly → daily  
- forward-filled  

---

##### (3) PCA Features (Derived)
- gt_pca_1  
- gt_pca_2  

👉 Compress multiple correlated search signals into:
- low-dimensional sentiment factors  

---

### 3. Data Alignment Logic

All datasets are transformed to **daily frequency**:

- Weekly data → forward-filled  
- Missing values → forward-filled  
- All merged on a common date index  

👉 Ensures:
- Consistent timeline  
- No gaps during modeling  

---

### 4. Target Variable (y)

The prediction target is:

- **Next-day direction**

Definition:
- 1 → price increases tomorrow  
- 0 → price decreases or stays the same  

👉 A binary classification problem  

---

### 5. Summary

This section transforms raw multi-source data into a **high-dimensional, structured feature matrix** by:

- Aligning all data to a daily timeline  
- Engineering technical, macro, and behavioral features  
- Creating regime and interaction signals  
- Reducing redundancy and noise  

👉 The result is a **clean and information-rich input dataset** ready for machine learning models  

In [18]:
# Cell 2 — Feature engineering + daily alignment
#
# This cell builds the main modeling table.
# It converts all sources to one daily timeline, applies the required fill/drop rules,
# creates technical / lagged / regime / interaction features, and defines the next-day target.
# The final outputs are `X_btc`, `X_eth`, `y_btc`, and `y_eth`.

import numpy as np  # numerical operations used throughout feature engineering


# Drop rows that are fully empty before stricter asset-level cleaning.
def _drop_all_missing_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(axis=0, how="all").copy()  # remove all-null rows and return a copy to avoid chained-assignment issues


# Build a complete daily calendar so all sources can be aligned on identical dates.
def make_daily_index(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return pd.DataFrame({"date": pd.date_range(start=start, end=end, freq="D")})  # one row per calendar day


# Convert a lower-frequency or irregularly spaced series into daily frequency by forward filling.
def to_daily_ffill(
    df: pd.DataFrame,
    date_col: str,
    value_cols: list[str] | None = None,
    start: pd.Timestamp | None = None,
    end: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """Resample to daily frequency and forward-fill values."""
    out = df.copy()  # work on a copy so the original input is not mutated
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")  # parse the date column to datetime
    out = out.dropna(subset=[date_col])  # remove rows where the date could not be parsed

    if value_cols is None:  # if no explicit value columns are given
        value_cols = [c for c in out.columns if c != date_col]  # use every non-date column as a value column

    out = out[[date_col] + value_cols].sort_values(date_col)  # keep only needed columns and sort by date

    if start is None:  # if start bound is not supplied
        start = out[date_col].min()  # start at the first available observation
    if end is None:  # if end bound is not supplied
        end = out[date_col].max()  # end at the last available observation

    idx = make_daily_index(start=start, end=end)  # create the full daily backbone
    out = idx.merge(out, left_on="date", right_on=date_col, how="left")  # left join to place sparse data on the daily calendar

    if date_col != "date":  # if the source date column has a different name than the canonical output date
        out = out.drop(columns=[date_col])  # drop the duplicate right-hand date column

    out[value_cols] = out[value_cols].ffill()  # carry the latest known value forward across missing days
    return out  # return a daily, forward-filled DataFrame


# Convert weekly Google Trends values into a daily series according to the assignment rule.
def google_trends_to_daily(
    df_weekly: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp,
    week_col: str = "Week",
    shift_days: int = 7,
) -> pd.DataFrame:
    """Google Trends is weekly; each point reflects the trend of the *following* week.

    We implement this by shifting the week timestamp forward by 7 days, then forward-filling to daily.
    """
    w = df_weekly.copy()  # work on a copy of the weekly trends table
    w.columns = [c.strip() for c in w.columns]  # normalize headers to avoid whitespace mismatches

    value_cols = [c for c in w.columns if c != week_col]  # find the non-date data column
    if len(value_cols) != 1:  # each trends file should contain exactly one series besides the week column
        raise ValueError(f"Expected 1 value column besides '{week_col}', got {value_cols}")

    value_col = value_cols[0]  # extract the single trends series name
    w[week_col] = pd.to_datetime(w[week_col], errors="coerce")  # parse weekly timestamps
    w = w.dropna(subset=[week_col])  # remove rows with invalid week values

    w[week_col] = w[week_col] + pd.Timedelta(days=shift_days)  # shift the weekly timestamp forward by 7 days per requirement

    daily = to_daily_ffill(  # convert the shifted weekly series into daily frequency
        w.rename(columns={week_col: "week"}),  # rename the week column to fit the helper signature cleanly
        date_col="week",  # use the shifted week as the date column
        value_cols=[value_col],  # only carry the actual trends value column
        start=start,  # align to the same global start date as the crypto series
        end=end,  # align to the same global end date as the crypto series
    )
    daily = daily.rename(columns={value_col: value_col.strip()})  # keep the trends column name clean after conversion
    return daily  # return the daily trends series


# Generate technical, lagged, regime, and interaction features from one crypto price/volume table.
def add_technical_features(prefix: str, df: pd.DataFrame) -> pd.DataFrame:
    """Assumes df has columns: date, price_close_usd, volume_24h_close_usd."""
    out = df.copy()  # do not overwrite the original asset table

    close = out["price_close_usd"].astype(float)  # use closing price as the base price series
    vol = out["volume_24h_close_usd"].astype(float)  # use 24h trading volume as the base volume series

    out[f"{prefix}_ret_1d"] = close.pct_change(1)  # 1-day simple return
    out[f"{prefix}_logret_1d"] = np.log(close).diff(1)  # 1-day log return
    out[f"{prefix}_ret_3d"] = close.pct_change(3)  # 3-day simple return
    out[f"{prefix}_ret_7d"] = close.pct_change(7)  # 7-day simple return
    out[f"{prefix}_ret_30d"] = close.pct_change(30)  # 30-day simple return used for regime detection

    out[f"{prefix}_return_lag1"] = out[f"{prefix}_ret_1d"].shift(1)  # yesterday's 1-day return
    out[f"{prefix}_return_lag2"] = out[f"{prefix}_ret_1d"].shift(2)  # two-day lag of 1-day return
    out[f"{prefix}_return_lag3"] = out[f"{prefix}_ret_1d"].shift(3)  # three-day lag of 1-day return

    out[f"{prefix}_volatility_7d"] = out[f"{prefix}_logret_1d"].rolling(7).std()  # 7-day rolling volatility
    out[f"{prefix}_volatility_14d"] = out[f"{prefix}_logret_1d"].rolling(14).std()  # 14-day rolling volatility
    out[f"{prefix}_volatility_30d"] = out[f"{prefix}_logret_1d"].rolling(30).std()  # 30-day rolling volatility
    out[f"{prefix}_ma_7d"] = close.rolling(7).mean()  # 7-day moving average of close
    out[f"{prefix}_ma_14d"] = close.rolling(14).mean()  # 14-day moving average of close
    out[f"{prefix}_ma_ratio_7_14"] = out[f"{prefix}_ma_7d"] / out[f"{prefix}_ma_14d"].replace(0, np.nan)  # short MA relative to medium MA

    out[f"{prefix}_vol_chg_1d"] = vol.pct_change(1)  # 1-day percentage change in volume
    out[f"{prefix}_vol_ma_7d"] = vol.rolling(7).mean()  # 7-day moving average of volume
    out[f"{prefix}_vol_ratio"] = vol / out[f"{prefix}_vol_ma_7d"].replace(0, np.nan)  # current volume relative to recent average
    out[f"{prefix}_vol_lag1"] = vol.shift(1)  # yesterday's raw volume
    out[f"{prefix}_volume_change_lag1"] = out[f"{prefix}_vol_chg_1d"].shift(1)  # yesterday's volume change

    delta = close.diff(1)  # day-to-day price change used in RSI construction
    gain = delta.where(delta > 0, 0.0)  # keep only positive moves for RSI gains
    loss = (-delta.where(delta < 0, 0.0))  # convert negative moves to positive magnitudes for RSI losses
    avg_gain = gain.rolling(14).mean()  # 14-day average gain
    avg_loss = loss.rolling(14).mean()  # 14-day average loss
    rs = avg_gain / (avg_loss.replace(0, np.nan))  # relative strength = avg gain / avg loss
    out[f"{prefix}_rsi_14"] = 100 - (100 / (1 + rs))  # convert relative strength into RSI scale
    out[f"{prefix}_RSI_lag1"] = out[f"{prefix}_rsi_14"].shift(1)  # yesterday's RSI

    ema12 = close.ewm(span=12, adjust=False).mean()  # short EMA for MACD
    ema26 = close.ewm(span=26, adjust=False).mean()  # long EMA for MACD
    macd = ema12 - ema26  # MACD line
    signal = macd.ewm(span=9, adjust=False).mean()  # MACD signal line
    out[f"{prefix}_macd"] = macd  # store MACD line
    out[f"{prefix}_macd_signal"] = signal  # store MACD signal line
    out[f"{prefix}_macd_hist"] = macd - signal  # MACD histogram = MACD - signal

    out[f"{prefix}_bull_market_dummy"] = (out[f"{prefix}_ret_30d"] > 0).astype(float)  # 1 if the 30-day return is positive
    out[f"{prefix}_volatility_regime"] = (
        out[f"{prefix}_volatility_14d"] > out[f"{prefix}_volatility_14d"].rolling(30).median()
    ).astype(float)  # 1 if current 14-day volatility is above its recent median

    out[f"{prefix}_momentum_x_volume"] = out[f"{prefix}_ret_7d"] * out[f"{prefix}_vol_chg_1d"]  # momentum-volume interaction
    out[f"{prefix}_ret1_x_bull"] = out[f"{prefix}_ret_1d"] * out[f"{prefix}_bull_market_dummy"]  # return active only in bull regime
    out[f"{prefix}_ret1_x_bear"] = out[f"{prefix}_ret_1d"] * (1 - out[f"{prefix}_bull_market_dummy"])  # return active only in bear regime

    return out  # return the original asset table plus all engineered features


def compress_google_trends_with_pca(df: pd.DataFrame, trend_cols: list[str], n_components: int = 2) -> pd.DataFrame:
    """Compress multiple Google Trends columns into a few PCA components."""
    if len(trend_cols) <= n_components:  # if there are too few trend columns, PCA would not reduce anything
        return df  # return the original DataFrame unchanged

    from sklearn.decomposition import PCA  # imported here because this helper is only used in feature building
    from sklearn.preprocessing import StandardScaler  # scale trends before PCA so high-variance columns do not dominate

    tmp = df.copy()  # work on a copy of the merged feature table
    valid_cols = [c for c in trend_cols if c in tmp.columns]  # keep only trend columns that actually exist
    if len(valid_cols) <= n_components:  # if valid trend count is too small, skip PCA safely
        return tmp

    X = tmp[valid_cols].copy()  # isolate the trend feature block
    X = X.ffill().bfill().fillna(0.0)  # fill any remaining gaps before PCA so decomposition will not fail
    X_scaled = StandardScaler().fit_transform(X)  # standardize trends to mean 0 / variance 1 before PCA

    pca = PCA(n_components=n_components, random_state=42)  # keep a small number of principal components
    comp = pca.fit_transform(X_scaled)  # compute the low-dimensional trend representation

    for i in range(n_components):  # add each principal component back to the full DataFrame
        tmp[f"gt_pca_{i+1}"] = comp[:, i]

    drop_cols = [c for c in valid_cols if "_main" in c]  # drop the dense asset-main GT columns after compression
    tmp = tmp.drop(columns=drop_cols, errors="ignore")  # keep GT PCA columns and simpler interpretable trend columns
    return tmp  # return the DataFrame with compressed trend features


# Remove one of any pair/group of very highly correlated columns to reduce redundancy and instability.
def drop_high_collinearity(df: pd.DataFrame, threshold: float = 0.98, protected: set[str] | None = None) -> pd.DataFrame:
    """Drop highly collinear numeric columns to improve stability for LR/SVM/MLP."""
    out = df.copy()  # work on a copy of the feature table
    if protected is None:  # if no protected set is provided
        protected = set()  # initialize an empty protected set

    num_cols = [c for c in out.columns if c != "date" and pd.api.types.is_numeric_dtype(out[c])]  # numeric columns eligible for correlation checks
    if len(num_cols) < 2:  # if there are fewer than two numeric columns, correlation pruning is unnecessary
        return out

    corr = out[num_cols].corr().abs()  # absolute correlation matrix so sign does not matter
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))  # keep only the upper triangle to avoid duplicate pairs

    to_drop = []  # collect redundant columns here
    for col in upper.columns:  # inspect one candidate column at a time
        if col in protected:  # never drop protected columns needed for target creation or key signals
            continue
        if any((upper[col] > threshold) & (~upper[col].isna())):  # if this column is too correlated with any earlier column
            to_drop.append(col)  # mark it for removal

    return out.drop(columns=to_drop, errors="ignore")  # drop redundant columns and return the pruned table


# Build the full modeling table for one asset, including features, target, and optional final unlabeled row.
def build_feature_table(asset: str, keep_last_unlabeled: bool = False) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """asset in {"btc", "eth"}.

    Returns (X_df_with_date, y, feature_cols).

    - If keep_last_unlabeled=False (default): X and y are aligned and contain only rows with known labels.
    - If keep_last_unlabeled=True: X includes the most recent date row (features for forecasting the next day),
      while y still excludes that last row.
    """

    # Select the asset-specific crypto table and matching search-interest tables.
    if asset == "btc":
        coin = datasets["btc_daily"].copy()  # raw BTC daily crypto data
        target_close_col = "price_close_usd"  # close column used to define the label
        gt_main = datasets["gt_bitcoin"].copy()  # BTC main Google Trends series
        gt_pm = datasets["gt_btc_pm"].copy()  # BTC price-manipulation trends series
    elif asset == "eth":
        coin = datasets["eth_daily"].copy()  # raw ETH daily crypto data
        target_close_col = "price_close_usd"  # close column used to define the label
        gt_main = datasets["gt_ethereum"].copy()  # ETH main Google Trends series
        gt_pm = datasets["gt_eth_pm"].copy()  # ETH price-manipulation trends series
    else:
        raise ValueError("asset must be 'btc' or 'eth'")  # reject unsupported asset names

    coin = _drop_all_missing_rows(coin)  # remove fully empty rows first
    coin = coin.dropna(axis=0, how="any").reset_index(drop=True)  # drop partially missing crypto rows per the assignment rule

    coin = coin.sort_values("date").reset_index(drop=True)  # ensure the asset data is sorted in time order
    start = coin["date"].min()  # first date used to build the common daily calendar
    end = coin["date"].max()  # last date used to build the common daily calendar

    effr_daily = to_daily_ffill(  # convert EFFR to daily frequency and forward fill missing days
        datasets["effr"].rename(columns={"observation_date": "date"}),
        date_col="date",
        value_cols=["EFFR"],
        start=start,
        end=end,
    ).rename(columns={"EFFR": "effr"})  # rename the macro column to a simpler feature name

    qqq_daily = to_daily_ffill(datasets["qqq"], date_col="Date", start=start, end=end)  # daily-aligned Nasdaq proxy
    uup_daily = to_daily_ffill(datasets["uup"], date_col="Date", start=start, end=end)  # daily-aligned dollar proxy
    gld_daily = to_daily_ffill(datasets["gld"], date_col="Date", start=start, end=end)  # daily-aligned gold proxy
    bsv_daily = to_daily_ffill(datasets["bsv"], date_col="Date", start=start, end=end)  # daily-aligned bond proxy

    def _prefix_cols(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
        out = df.copy()  # avoid mutating the input market DataFrame
        for c in list(out.columns):  # inspect each column name
            if c != "date":  # keep the shared date column unprefixed
                out = out.rename(columns={c: f"{prefix}_{c}"})  # prefix all value columns by source name
        return out  # return the renamed DataFrame

    qqq_daily = _prefix_cols(qqq_daily, "qqq")  # add qqq_ prefix to market columns
    uup_daily = _prefix_cols(uup_daily, "uup")  # add uup_ prefix to market columns
    gld_daily = _prefix_cols(gld_daily, "gld")  # add gld_ prefix to market columns
    bsv_daily = _prefix_cols(bsv_daily, "bsv")  # add bsv_ prefix to market columns

    gt_crypto = datasets["gt_crypto"].copy()  # generic cryptocurrency search-interest series

    gt_main_daily = google_trends_to_daily(gt_main, start=start, end=end)  # convert asset main trends to daily
    gt_pm_daily = google_trends_to_daily(gt_pm, start=start, end=end)  # convert price-manipulation trends to daily
    gt_crypto_daily = google_trends_to_daily(gt_crypto, start=start, end=end)  # convert generic crypto trends to daily

    gt_main_daily = gt_main_daily.rename(columns={c: f"gt_{asset}_main" for c in gt_main_daily.columns if c != "date"})  # prefix asset main GT column
    gt_pm_daily = gt_pm_daily.rename(columns={c: f"gt_{asset}_pm" for c in gt_pm_daily.columns if c != "date"})  # prefix asset PM GT column
    gt_crypto_daily = gt_crypto_daily.rename(columns={c: "gt_crypto" for c in gt_crypto_daily.columns if c != "date"})  # standardize generic crypto GT name

    coin_feat = add_technical_features(asset, coin)  # add technical/regime/lagged features to the crypto table

    base = make_daily_index(start=start, end=end)  # create the global daily date backbone for this asset window
    merged = base.merge(coin_feat, on="date", how="left")  # anchor the dataset on crypto features first

    merged = merged.dropna(axis=0, how="any").reset_index(drop=True)  # remove dates that lack the anchor crypto data entirely

    merged = merged.merge(effr_daily, on="date", how="left")  # merge macro features
    merged = merged.merge(qqq_daily, on="date", how="left")  # merge Nasdaq proxy features
    merged = merged.merge(uup_daily, on="date", how="left")  # merge dollar proxy features
    merged = merged.merge(gld_daily, on="date", how="left")  # merge gold proxy features
    merged = merged.merge(bsv_daily, on="date", how="left")  # merge bond proxy features
    merged = merged.merge(gt_main_daily, on="date", how="left")  # merge asset main search-interest feature
    merged = merged.merge(gt_pm_daily, on="date", how="left")  # merge price-manipulation search-interest feature
    merged = merged.merge(gt_crypto_daily, on="date", how="left")  # merge generic crypto search-interest feature

    ext_cols = [  # define external columns that should be forward-filled after merges
        c
        for c in merged.columns
        if c not in {"date"}  # never treat the date column as a fill target
        and not c.startswith("price_")  # keep raw price columns separate from external fill logic
        and not c.startswith("market_cap_")  # keep raw market-cap columns separate from external fill logic
        and not c.startswith("volume_24h_")  # keep raw crypto volume columns separate from external fill logic
        and not c.startswith(asset + "_")  # exclude asset-native engineered features from this external fill list
    ]
    merged[ext_cols] = merged[ext_cols].ffill()  # forward-fill macro/market/trend features across missing dates

    if "effr" in merged.columns:  # only create the funding proxy if the macro series exists
        merged[f"{asset}_funding_lag1"] = merged["effr"].shift(1)  # use lagged EFFR as a simple funding-cost proxy

    merged[f"{asset}_bull_market_dummy"] = (merged[f"{asset}_ret_30d"] > 0).astype(float)  # 1 when the asset 30-day return is positive
    if "qqq_Close" in merged.columns:  # only compute the volatility-regime proxy if QQQ close is available
        qqq_ret = pd.to_numeric(merged["qqq_Close"], errors="coerce").pct_change(1)  # 1-day QQQ return series
        qqq_vol_20 = qqq_ret.rolling(20).std()  # 20-day realized volatility proxy for QQQ
        merged["VIX_high_regime"] = (qqq_vol_20 > qqq_vol_20.rolling(60).median()).astype(float)  # high-vol regime flag
        merged["qqq_ret_1d"] = qqq_ret  # store QQQ return explicitly for interaction features
    else:
        merged["VIX_high_regime"] = 0.0  # fallback if QQQ data is unavailable
        merged["qqq_ret_1d"] = 0.0  # fallback if QQQ data is unavailable

    if "uup_Close" in merged.columns:  # only compute dollar-trend regime if UUP close is available
        uup_ma_20 = pd.to_numeric(merged["uup_Close"], errors="coerce").rolling(20).mean()  # short UUP moving average
        uup_ma_60 = pd.to_numeric(merged["uup_Close"], errors="coerce").rolling(60).mean()  # longer UUP moving average
        merged["DXY_trend_regime"] = (uup_ma_20 > uup_ma_60).astype(float)  # 1 when short MA is above long MA
    else:
        merged["DXY_trend_regime"] = 0.0  # fallback if UUP data is unavailable

    merged[f"{asset}_ret1_x_vix"] = merged[f"{asset}_ret_1d"] * merged["VIX_high_regime"]  # asset return × high-vol regime
    merged[f"{asset}_ret1_x_qqqret"] = merged[f"{asset}_ret_1d"] * merged["qqq_ret_1d"]  # asset return × QQQ return
    merged[f"{asset}_momentum_x_volume2"] = merged[f"{asset}_ret_3d"] * merged[f"{asset}_vol_chg_1d"]  # 3-day momentum × volume change
    merged[f"{asset}_rsi_x_vol"] = merged[f"{asset}_rsi_14"] * merged[f"{asset}_volatility_14d"]  # RSI × volatility interaction
    if "gld_Close" in merged.columns:  # only create gold interaction if GLD is available
        gld_ret = pd.to_numeric(merged["gld_Close"], errors="coerce").pct_change(1)  # 1-day gold return
        merged[f"{asset}_ret_x_gldret"] = merged[f"{asset}_ret_1d"] * gld_ret  # asset return × gold return

    bull = merged[f"{asset}_bull_market_dummy"]  # convenience alias for the asset bull-regime dummy
    for base_feat in [f"{asset}_ret_1d", f"{asset}_ret_7d", f"{asset}_vol_chg_1d", f"{asset}_rsi_14"]:  # selected core signals to split by regime
        merged[f"{base_feat}_x_bull"] = merged[base_feat] * bull  # feature active only in bull regime
        merged[f"{base_feat}_x_bear"] = merged[base_feat] * (1 - bull)  # feature active only in bear regime

    for c in merged.columns:  # convert every non-date column to numeric where possible
        if c != "date":
            merged[c] = pd.to_numeric(merged[c], errors="coerce")  # invalid strings become NaN

    merged = merged.replace([np.inf, -np.inf], np.nan)  # remove infinite values created by ratios or percentage changes

    merged[ext_cols] = merged[ext_cols].ffill()  # forward-fill external features again after numeric coercion/replacement

    numeric_cols = [c for c in merged.columns if c != "date"]  # all numeric-like columns except the date
    for c in numeric_cols:  # clip each column individually to its robust percentile band
        lo, hi = merged[c].quantile(0.01), merged[c].quantile(0.99)  # lower and upper clipping thresholds
        merged[c] = merged[c].clip(lower=lo, upper=hi)  # reduce extreme outliers without fully discarding the row

    trend_cols = [c for c in merged.columns if c.startswith("gt_")]  # collect all Google Trends-derived columns
    merged = compress_google_trends_with_pca(merged, trend_cols=trend_cols, n_components=2)  # compress dense GT block into PCA factors

    protected_cols = {  # columns that should never be dropped during correlation pruning
        target_close_col,
        "volume_24h_close_usd",
        f"{asset}_ret_1d",
        f"{asset}_ret_7d",
        f"{asset}_rsi_14",
        "gt_pca_1",
        "gt_pca_2",
    }

    merged = drop_high_collinearity(  # prune redundant columns while preserving protected core features
        merged,
        threshold=0.98,
        protected=protected_cols,
    )

    merged = merged.dropna(axis=0, how="any").reset_index(drop=True)  # remove any rows that still contain missing values

    if target_close_col not in merged.columns:  # verify the target source survived all preprocessing steps
        raise RuntimeError(f"Required target source column missing: {target_close_col}")

    merged[f"{asset}_target_up"] = (  # define the label using the next day's close vs today's close
        merged[target_close_col].shift(-1) > merged[target_close_col]
    ).astype(int)

    y = merged[f"{asset}_target_up"].iloc[:-1].astype(int).reset_index(drop=True)  # drop the last unlabeled row from y because it has no next day

    drop_cols = {  # columns excluded from the feature matrix
        "date",
        f"{asset}_target_up",
    }

    feature_cols = [c for c in merged.columns if c not in drop_cols]  # final list of usable feature columns

    X_all = merged[["date"] + feature_cols].copy().reset_index(drop=True)  # keep date for later time-based splitting and forecasting

    if not keep_last_unlabeled:  # default mode used for model development
        X_all = X_all.iloc[:-1].reset_index(drop=True)  # remove the last row because its label is unknown

    return X_all, y, feature_cols  # return features-with-date, labels, and feature column names


X_btc, y_btc, feat_btc = build_feature_table("btc")  # build the full BTC modeling table
X_eth, y_eth, feat_eth = build_feature_table("eth")  # build the full ETH modeling table

X_btc.head(), X_btc.shape, y_btc.value_counts(normalize=True).round(3)  # preview BTC features, shape, and class balance

(        date  day_start_ts_s  price_open_usd  price_close_usd  volume_24h_close_usd  btc_ret_1d  btc_ret_3d  btc_ret_7d  btc_ret_30d  \
 0 2025-02-14    1.739694e+09    96666.709608     97190.842815          3.231509e+10    0.008474    0.017048    0.011264    -0.025415   
 1 2025-02-15    1.739694e+09    97525.564328     97545.416081          1.468093e+10    0.003648   -0.003803    0.010969    -0.023350   
 2 2025-02-16    1.739694e+09    97651.637793     96889.888850          1.239465e+10   -0.006720    0.005351    0.010942    -0.071469   
 3 2025-02-17    1.739750e+09    96122.866005     95910.634400          2.662055e+10   -0.010107   -0.013172   -0.015195    -0.079702   
 4 2025-02-18    1.739837e+09    95784.629539     95259.617920          3.499152e+10   -0.006788   -0.023433   -0.003161    -0.077137   
 
    btc_return_lag1  btc_return_lag2  btc_return_lag3  btc_volatility_7d  btc_volatility_14d  btc_volatility_30d  btc_ma_ratio_7_14  \
 0        -0.015764         0.024655     

### Section 2.1 — BTC Prediction Model: Inputs and Outputs


In [ ]:
from IPython.display import HTML, display

html = r'''
<div id="mermaid-diagram-2" class="mermaid">
flowchart LR
    A[BTC next-day direction prediction framework]
    A --> B[Input feature groups]

    B --> B1[Native BTC market features]
    B --> B2[External market context]
    B --> B3[Search-attention features]

    B1 --> C1[Raw market state]
    B1 --> C2[Returns and lagged signals]
    B1 --> C3[Volatility and trend indicators]
    B1 --> C4[Volume and liquidity measures]
    B1 --> C5[Oscillators and momentum indicators]
    B1 --> C6[BTC regime variables]

    B2 --> D1[Interest-rate proxy]
    B2 --> D2[Equity risk proxy: QQQ]
    B2 --> D3[Dollar proxy: UUP]
    B2 --> D4[Gold proxy: GLD]
    B2 --> D5[Bond proxy: BSV]
    B2 --> D6[External regime indicators]

    B3 --> E1[BTC manipulation search interest]
    B3 --> E2[Generic crypto search interest]
    B3 --> E3[Compressed BTC search factors]

    C1 --> F[Output]
    C2 --> F
    C3 --> F
    C4 --> F
    C5 --> F
    C6 --> F
    D1 --> F
    D2 --> F
    D3 --> F
    D4 --> F
    D5 --> F
    D6 --> F
    E1 --> F
    E2 --> F
    E3 --> F

    F --> G1[Predicted class: up vs. not up]
    F --> G2[Positive-class probability score]
    F --> G3[Interpretation: next-day BTC direction]
</div>

<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
  mermaid.initialize({ startOnLoad: false });
  await mermaid.run({
    querySelector: '#mermaid-diagram-2'
  });
</script>
'''
display(HTML(html))

**Complete BTC Input Feature Catalog**

<div align="center">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Feature Block</th>
      <th>Subcategory</th>
      <th>Feature</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="4">Native BTC market</td>
      <td>Raw timestamp</td>
      <td><code>day_start_ts_s</code></td>
      <td style="text-align: left;">Unix timestamp in seconds for the start of the daily BTC bar.</td>
    </tr>
    <tr>
      <td rowspan="2">Raw price level</td>
      <td><code>price_open_usd</code></td>
      <td style="text-align: left;">BTC opening price in USD for the day.</td>
    </tr>
    <tr>
      <td><code>price_close_usd</code></td>
      <td style="text-align: left;">BTC closing price in USD for the day.</td>
    </tr>
    <tr>
      <td>Raw volume level</td>
      <td><code>volume_24h_close_usd</code></td>
      <td style="text-align: left;">BTC 24-hour traded volume measured in USD near the close.</td>
    </tr>
    <tr>
      <td rowspan="7">BTC returns</td>
      <td rowspan="4">Simple returns</td>
      <td><code>btc_ret_1d</code></td>
      <td style="text-align: left;">1-day BTC simple return.</td>
    </tr>
    <tr>
      <td><code>btc_ret_3d</code></td>
      <td style="text-align: left;">3-day BTC simple return.</td>
    </tr>
    <tr>
      <td><code>btc_ret_7d</code></td>
      <td style="text-align: left;">7-day BTC simple return.</td>
    </tr>
    <tr>
      <td><code>btc_ret_30d</code></td>
      <td style="text-align: left;">30-day BTC simple return, also used for regime detection.</td>
    </tr>
    <tr>
      <td rowspan="3">Lagged returns</td>
      <td><code>btc_return_lag1</code></td>
      <td style="text-align: left;">Yesterday's 1-day BTC return.</td>
    </tr>
    <tr>
      <td><code>btc_return_lag2</code></td>
      <td style="text-align: left;">BTC 1-day return lagged by 2 days.</td>
    </tr>
    <tr>
      <td><code>btc_return_lag3</code></td>
      <td style="text-align: left;">BTC 1-day return lagged by 3 days.</td>
    </tr>
    <tr>
      <td rowspan="3">BTC risk</td>
      <td rowspan="3">Rolling volatility</td>
      <td><code>btc_volatility_7d</code></td>
      <td style="text-align: left;">7-day rolling volatility of BTC log returns.</td>
    </tr>
    <tr>
      <td><code>btc_volatility_14d</code></td>
      <td style="text-align: left;">14-day rolling volatility of BTC log returns.</td>
    </tr>
    <tr>
      <td><code>btc_volatility_30d</code></td>
      <td style="text-align: left;">30-day rolling volatility of BTC log returns.</td>
    </tr>
    <tr>
      <td>BTC trend</td>
      <td>Moving averages</td>
      <td><code>btc_ma_ratio_7_14</code></td>
      <td style="text-align: left;">Ratio of 7-day BTC moving average to 14-day BTC moving average.</td>
    </tr>
    <tr>
      <td rowspan="5">BTC volume</td>
      <td>Volume change</td>
      <td><code>btc_vol_chg_1d</code></td>
      <td style="text-align: left;">1-day percentage change in BTC trading volume.</td>
    </tr>
    <tr>
      <td>Volume trend</td>
      <td><code>btc_vol_ma_7d</code></td>
      <td style="text-align: left;">7-day moving average of BTC trading volume.</td>
    </tr>
    <tr>
      <td>Relative volume</td>
      <td><code>btc_vol_ratio</code></td>
      <td style="text-align: left;">Current BTC volume divided by its 7-day average volume.</td>
    </tr>
    <tr>
      <td>Lagged volume</td>
      <td><code>btc_vol_lag1</code></td>
      <td style="text-align: left;">Yesterday's raw BTC volume.</td>
    </tr>
    <tr>
      <td>Lagged volume change</td>
      <td><code>btc_volume_change_lag1</code></td>
      <td style="text-align: left;">Yesterday's BTC volume percentage change.</td>
    </tr>
    <tr>
      <td rowspan="5">BTC momentum oscillator</td>
      <td rowspan="2">RSI</td>
      <td><code>btc_rsi_14</code></td>
      <td style="text-align: left;">14-day relative strength index for BTC.</td>
    </tr>
    <tr>
      <td><code>btc_RSI_lag1</code></td>
      <td style="text-align: left;">Yesterday's BTC RSI value.</td>
    </tr>
    <tr>
      <td rowspan="3">MACD</td>
      <td><code>btc_macd</code></td>
      <td style="text-align: left;">BTC MACD line from 12/26-day exponential moving averages.</td>
    </tr>
    <tr>
      <td><code>btc_macd_signal</code></td>
      <td style="text-align: left;">Signal line of BTC MACD.</td>
    </tr>
    <tr>
      <td><code>btc_macd_hist</code></td>
      <td style="text-align: left;">BTC MACD histogram, equal to MACD minus signal line.</td>
    </tr>
    <tr>
      <td rowspan="2">BTC regime</td>
      <td>Price regime</td>
      <td><code>btc_bull_market_dummy</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when BTC 30-day return is positive.</td>
    </tr>
    <tr>
      <td>Volatility regime</td>
      <td><code>btc_volatility_regime</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when current 14-day BTC volatility is above its recent median.</td>
    </tr>
    <tr>
      <td>Macro</td>
      <td>Interest-rate level</td>
      <td><code>effr</code></td>
      <td style="text-align: left;">Effective Federal Funds Rate, used as a short-rate macro feature.</td>
    </tr>
    <tr>
      <td rowspan="9">External market</td>
      <td rowspan="3">Equity proxy</td>
      <td><code>qqq_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>QQQ</code>, used as a Nasdaq risk proxy.</td>
    </tr>
    <tr>
      <td><code>qqq_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>QQQ</code>.</td>
    </tr>
    <tr>
      <td><code>qqq_ret_1d</code></td>
      <td style="text-align: left;">1-day return of <code>QQQ</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">Dollar proxy</td>
      <td><code>uup_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>UUP</code>, used as a US dollar trend proxy.</td>
    </tr>
    <tr>
      <td><code>uup_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>UUP</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">Gold proxy</td>
      <td><code>gld_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>GLD</code>, used as a gold-market proxy.</td>
    </tr>
    <tr>
      <td><code>gld_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>GLD</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">Bond proxy</td>
      <td><code>bsv_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>BSV</code>, used as a bond-market proxy.</td>
    </tr>
    <tr>
      <td><code>bsv_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>BSV</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">External regime</td>
      <td>Equity-volatility regime</td>
      <td><code>VIX_high_regime</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when realized QQQ volatility is above its rolling median, acting as a VIX-like high-risk regime.</td>
    </tr>
    <tr>
      <td>Dollar-trend regime</td>
      <td><code>DXY_trend_regime</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when short-term <code>UUP</code> trend is above long-term <code>UUP</code> trend.</td>
    </tr>
    <tr>
      <td rowspan="4">Search attention</td>
      <td>BTC-specific query</td>
      <td><code>gt_btc_pm</code></td>
      <td style="text-align: left;">Google Trends signal for BTC price-manipulation-related search interest.</td>
    </tr>
    <tr>
      <td>Generic crypto query</td>
      <td><code>gt_crypto</code></td>
      <td style="text-align: left;">Google Trends signal for generic cryptocurrency search interest.</td>
    </tr>
    <tr>
      <td rowspan="2">Compressed BTC trend factors</td>
      <td><code>gt_pca_1</code></td>
      <td style="text-align: left;">First principal component summarizing the broader BTC Google Trends block after dimensionality reduction.</td>
    </tr>
    <tr>
      <td><code>gt_pca_2</code></td>
      <td style="text-align: left;">Second principal component summarizing the broader BTC Google Trends block after dimensionality reduction.</td>
    </tr>
    <tr>
      <td rowspan="11">BTC interaction</td>
      <td rowspan="2">Native interaction</td>
      <td><code>btc_momentum_x_volume</code></td>
      <td style="text-align: left;">BTC 7-day return multiplied by 1-day BTC volume change.</td>
    </tr>
    <tr>
      <td><code>btc_momentum_x_volume2</code></td>
      <td style="text-align: left;">BTC 3-day return multiplied by 1-day BTC volume change.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned return</td>
      <td><code>btc_ret1_x_bull</code></td>
      <td style="text-align: left;">BTC 1-day return active only in BTC bull-regime periods.</td>
    </tr>
    <tr>
      <td><code>btc_ret1_x_bear</code></td>
      <td style="text-align: left;">BTC 1-day return active only in BTC bear-regime periods.</td>
    </tr>
    <tr>
      <td>Oscillator x risk</td>
      <td><code>btc_rsi_x_vol</code></td>
      <td style="text-align: left;">BTC RSI multiplied by 14-day BTC volatility.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned momentum</td>
      <td><code>btc_ret_7d_x_bull</code></td>
      <td style="text-align: left;">BTC 7-day return active only in BTC bull regimes.</td>
    </tr>
    <tr>
      <td><code>btc_ret_7d_x_bear</code></td>
      <td style="text-align: left;">BTC 7-day return active only in BTC bear regimes.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned volume change</td>
      <td><code>btc_vol_chg_1d_x_bull</code></td>
      <td style="text-align: left;">BTC 1-day volume change active only in BTC bull regimes.</td>
    </tr>
    <tr>
      <td><code>btc_vol_chg_1d_x_bear</code></td>
      <td style="text-align: left;">BTC 1-day volume change active only in BTC bear regimes.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned RSI</td>
      <td><code>btc_rsi_14_x_bull</code></td>
      <td style="text-align: left;">BTC RSI active only in BTC bull regimes.</td>
    </tr>
    <tr>
      <td><code>btc_rsi_14_x_bear</code></td>
      <td style="text-align: left;">BTC RSI active only in BTC bear regimes.</td>
    </tr>
    <tr>
      <td rowspan="3">Cross-market interaction</td>
      <td>BTC x equity-vol regime</td>
      <td><code>btc_ret1_x_vix</code></td>
      <td style="text-align: left;">BTC 1-day return multiplied by the high-volatility market regime flag.</td>
    </tr>
    <tr>
      <td>BTC x equity return</td>
      <td><code>btc_ret1_x_qqqret</code></td>
      <td style="text-align: left;">BTC 1-day return multiplied by 1-day <code>QQQ</code> return.</td>
    </tr>
    <tr>
      <td>BTC x gold return</td>
      <td><code>btc_ret_x_gldret</code></td>
      <td style="text-align: left;">BTC 1-day return multiplied by 1-day <code>GLD</code> return.</td>
    </tr>
  </tbody>
</table>
</div>

### Section 2.2 — ETH Prediction Model: Inputs and Outputs

In [ ]:
from IPython.display import HTML, display

html = r'''
<div id="mermaid-eth-diagram" class="mermaid">
flowchart LR
    A[ETH prediction model input]

    A --> B[Input feature groups]

    B --> B1[Native ETH market features]
    B --> B2[External market context]
    B --> B3[Search-attention features]

    B1 --> C1[Raw market state]
    B1 --> C2[Returns and lags]
    B1 --> C3[Volatility and trend]
    B1 --> C4[Volume and liquidity]
    B1 --> C5[Oscillators and momentum]
    B1 --> C6[ETH regime features]

    B2 --> D1[Interest-rate proxy]
    B2 --> D2[Equity risk proxy: QQQ]
    B2 --> D3[Dollar proxy: UUP]
    B2 --> D4[Gold proxy: GLD]
    B2 --> D5[Bond proxy: BSV]
    B2 --> D6[External regime flags]

    B3 --> E1[ETH manipulation search interest]
    B3 --> E2[Generic crypto search interest]
    B3 --> E3[Compressed ETH search factors]

    C1 --> F[Output]
    C2 --> F
    C3 --> F
    C4 --> F
    C5 --> F
    C6 --> F
    D1 --> F
    D2 --> F
    D3 --> F
    D4 --> F
    D5 --> F
    D6 --> F
    E1 --> F
    E2 --> F
    E3 --> F

    F --> G1[Predicted class: up or not up]
    F --> G2[Positive-class score / probability]
    F --> G3[Meaning: next-day ETH direction]
</div>

<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
  mermaid.initialize({ startOnLoad: false });
  await mermaid.run({
    querySelector: '#mermaid-eth-diagram'
  });
</script>
'''

display(HTML(html))

**Complete ETH Input Feature Catalog**

<div align="center">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Feature Block</th>
      <th>Subcategory</th>
      <th>Feature</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="4">Native ETH market</td>
      <td>Raw timestamp</td>
      <td><code>day_start_ts_s</code></td>
      <td style="text-align: left;">Unix timestamp in seconds for the start of the daily ETH bar.</td>
    </tr>
    <tr>
      <td rowspan="2">Raw price level</td>
      <td><code>price_open_usd</code></td>
      <td style="text-align: left;">ETH opening price in USD for the day.</td>
    </tr>
    <tr>
      <td><code>price_close_usd</code></td>
      <td style="text-align: left;">ETH closing price in USD for the day.</td>
    </tr>
    <tr>
      <td>Raw volume level</td>
      <td><code>volume_24h_close_usd</code></td>
      <td style="text-align: left;">ETH 24-hour traded volume measured in USD near the close.</td>
    </tr>
    <tr>
      <td rowspan="7">ETH returns</td>
      <td rowspan="4">Simple returns</td>
      <td><code>eth_ret_1d</code></td>
      <td style="text-align: left;">1-day ETH simple return.</td>
    </tr>
    <tr>
      <td><code>eth_ret_3d</code></td>
      <td style="text-align: left;">3-day ETH simple return.</td>
    </tr>
    <tr>
      <td><code>eth_ret_7d</code></td>
      <td style="text-align: left;">7-day ETH simple return.</td>
    </tr>
    <tr>
      <td><code>eth_ret_30d</code></td>
      <td style="text-align: left;">30-day ETH simple return, also used for regime detection.</td>
    </tr>
    <tr>
      <td rowspan="3">Lagged returns</td>
      <td><code>eth_return_lag1</code></td>
      <td style="text-align: left;">Yesterday's 1-day ETH return.</td>
    </tr>
    <tr>
      <td><code>eth_return_lag2</code></td>
      <td style="text-align: left;">ETH 1-day return lagged by 2 days.</td>
    </tr>
    <tr>
      <td><code>eth_return_lag3</code></td>
      <td style="text-align: left;">ETH 1-day return lagged by 3 days.</td>
    </tr>
    <tr>
      <td rowspan="3">ETH risk</td>
      <td rowspan="3">Rolling volatility</td>
      <td><code>eth_volatility_7d</code></td>
      <td style="text-align: left;">7-day rolling volatility of ETH log returns.</td>
    </tr>
    <tr>
      <td><code>eth_volatility_14d</code></td>
      <td style="text-align: left;">14-day rolling volatility of ETH log returns.</td>
    </tr>
    <tr>
      <td><code>eth_volatility_30d</code></td>
      <td style="text-align: left;">30-day rolling volatility of ETH log returns.</td>
    </tr>
    <tr>
      <td>ETH trend</td>
      <td>Moving averages</td>
      <td><code>eth_ma_ratio_7_14</code></td>
      <td style="text-align: left;">Ratio of 7-day ETH moving average to 14-day ETH moving average.</td>
    </tr>
    <tr>
      <td rowspan="5">ETH volume</td>
      <td>Volume change</td>
      <td><code>eth_vol_chg_1d</code></td>
      <td style="text-align: left;">1-day percentage change in ETH trading volume.</td>
    </tr>
    <tr>
      <td>Volume trend</td>
      <td><code>eth_vol_ma_7d</code></td>
      <td style="text-align: left;">7-day moving average of ETH trading volume.</td>
    </tr>
    <tr>
      <td>Relative volume</td>
      <td><code>eth_vol_ratio</code></td>
      <td style="text-align: left;">Current ETH volume divided by its 7-day average volume.</td>
    </tr>
    <tr>
      <td>Lagged volume</td>
      <td><code>eth_vol_lag1</code></td>
      <td style="text-align: left;">Yesterday's raw ETH volume.</td>
    </tr>
    <tr>
      <td>Lagged volume change</td>
      <td><code>eth_volume_change_lag1</code></td>
      <td style="text-align: left;">Yesterday's ETH volume percentage change.</td>
    </tr>
    <tr>
      <td rowspan="5">ETH momentum oscillator</td>
      <td rowspan="2">RSI</td>
      <td><code>eth_rsi_14</code></td>
      <td style="text-align: left;">14-day relative strength index for ETH.</td>
    </tr>
    <tr>
      <td><code>eth_RSI_lag1</code></td>
      <td style="text-align: left;">Yesterday's ETH RSI value.</td>
    </tr>
    <tr>
      <td rowspan="3">MACD</td>
      <td><code>eth_macd</code></td>
      <td style="text-align: left;">ETH MACD line from 12/26-day exponential moving averages.</td>
    </tr>
    <tr>
      <td><code>eth_macd_signal</code></td>
      <td style="text-align: left;">Signal line of ETH MACD.</td>
    </tr>
    <tr>
      <td><code>eth_macd_hist</code></td>
      <td style="text-align: left;">ETH MACD histogram, equal to MACD minus signal line.</td>
    </tr>
    <tr>
      <td rowspan="2">ETH regime</td>
      <td>Price regime</td>
      <td><code>eth_bull_market_dummy</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when ETH 30-day return is positive.</td>
    </tr>
    <tr>
      <td>Volatility regime</td>
      <td><code>eth_volatility_regime</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when current 14-day ETH volatility is above its recent median.</td>
    </tr>
    <tr>
      <td>Macro</td>
      <td>Interest-rate level</td>
      <td><code>effr</code></td>
      <td style="text-align: left;">Effective Federal Funds Rate, used as a short-rate macro feature.</td>
    </tr>
    <tr>
      <td rowspan="9">External market</td>
      <td rowspan="3">Equity proxy</td>
      <td><code>qqq_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>QQQ</code>, used as a Nasdaq risk proxy.</td>
    </tr>
    <tr>
      <td><code>qqq_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>QQQ</code>.</td>
    </tr>
    <tr>
      <td><code>qqq_ret_1d</code></td>
      <td style="text-align: left;">1-day return of <code>QQQ</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">Dollar proxy</td>
      <td><code>uup_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>UUP</code>, used as a US dollar trend proxy.</td>
    </tr>
    <tr>
      <td><code>uup_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>UUP</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">Gold proxy</td>
      <td><code>gld_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>GLD</code>, used as a gold-market proxy.</td>
    </tr>
    <tr>
      <td><code>gld_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>GLD</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">Bond proxy</td>
      <td><code>bsv_Open</code></td>
      <td style="text-align: left;">Daily opening price of <code>BSV</code>, used as a bond-market proxy.</td>
    </tr>
    <tr>
      <td><code>bsv_Volume</code></td>
      <td style="text-align: left;">Daily trading volume of <code>BSV</code>.</td>
    </tr>
    <tr>
      <td rowspan="2">External regime</td>
      <td>Equity-volatility regime</td>
      <td><code>VIX_high_regime</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when realized QQQ volatility is above its rolling median, acting as a VIX-like high-risk regime.</td>
    </tr>
    <tr>
      <td>Dollar-trend regime</td>
      <td><code>DXY_trend_regime</code></td>
      <td style="text-align: left;">Binary flag equal to 1 when short-term <code>UUP</code> trend is above long-term <code>UUP</code> trend.</td>
    </tr>
    <tr>
      <td rowspan="4">Search attention</td>
      <td>ETH-specific query</td>
      <td><code>gt_eth_pm</code></td>
      <td style="text-align: left;">Google Trends signal for ETH price-manipulation-related search interest.</td>
    </tr>
    <tr>
      <td>Generic crypto query</td>
      <td><code>gt_crypto</code></td>
      <td style="text-align: left;">Google Trends signal for generic cryptocurrency search interest.</td>
    </tr>
    <tr>
      <td rowspan="2">Compressed ETH trend factors</td>
      <td><code>gt_pca_1</code></td>
      <td style="text-align: left;">First principal component summarizing the broader ETH Google Trends block after dimensionality reduction.</td>
    </tr>
    <tr>
      <td><code>gt_pca_2</code></td>
      <td style="text-align: left;">Second principal component summarizing the broader ETH Google Trends block after dimensionality reduction.</td>
    </tr>
    <tr>
      <td rowspan="11">ETH interaction</td>
      <td rowspan="2">Native interaction</td>
      <td><code>eth_momentum_x_volume</code></td>
      <td style="text-align: left;">ETH 7-day return multiplied by 1-day ETH volume change.</td>
    </tr>
    <tr>
      <td><code>eth_momentum_x_volume2</code></td>
      <td style="text-align: left;">ETH 3-day return multiplied by 1-day ETH volume change.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned return</td>
      <td><code>eth_ret1_x_bull</code></td>
      <td style="text-align: left;">ETH 1-day return active only in ETH bull-regime periods.</td>
    </tr>
    <tr>
      <td><code>eth_ret1_x_bear</code></td>
      <td style="text-align: left;">ETH 1-day return active only in ETH bear-regime periods.</td>
    </tr>
    <tr>
      <td>Oscillator x risk</td>
      <td><code>eth_rsi_x_vol</code></td>
      <td style="text-align: left;">ETH RSI multiplied by 14-day ETH volatility.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned momentum</td>
      <td><code>eth_ret_7d_x_bull</code></td>
      <td style="text-align: left;">ETH 7-day return active only in ETH bull regimes.</td>
    </tr>
    <tr>
      <td><code>eth_ret_7d_x_bear</code></td>
      <td style="text-align: left;">ETH 7-day return active only in ETH bear regimes.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned volume change</td>
      <td><code>eth_vol_chg_1d_x_bull</code></td>
      <td style="text-align: left;">ETH 1-day volume change active only in ETH bull regimes.</td>
    </tr>
    <tr>
      <td><code>eth_vol_chg_1d_x_bear</code></td>
      <td style="text-align: left;">ETH 1-day volume change active only in ETH bear regimes.</td>
    </tr>
    <tr>
      <td rowspan="2">Regime-conditioned RSI</td>
      <td><code>eth_rsi_14_x_bull</code></td>
      <td style="text-align: left;">ETH RSI active only in ETH bull regimes.</td>
    </tr>
    <tr>
      <td><code>eth_rsi_14_x_bear</code></td>
      <td style="text-align: left;">ETH RSI active only in ETH bear regimes.</td>
    </tr>
    <tr>
      <td rowspan="3">Cross-market interaction</td>
      <td>ETH x equity-vol regime</td>
      <td><code>eth_ret1_x_vix</code></td>
      <td style="text-align: left;">ETH 1-day return multiplied by the high-volatility market regime flag.</td>
    </tr>
    <tr>
      <td>ETH x equity return</td>
      <td><code>eth_ret1_x_qqqret</code></td>
      <td style="text-align: left;">ETH 1-day return multiplied by 1-day <code>QQQ</code> return.</td>
    </tr>
    <tr>
      <td>ETH x gold return</td>
      <td><code>eth_ret_x_gldret</code></td>
      <td style="text-align: left;">ETH 1-day return multiplied by 1-day <code>GLD</code> return.</td>
    </tr>
  </tbody>
</table>
</div>

## Section 3.1 — Time-Based Train / Validation / Test Split

### 1. What This Section Does

This section implements the **data splitting layer** of the pipeline.

Its purpose is to:
- Split the dataset into **training, validation, and test sets**
- Ensure **strict chronological order**
- Apply final **data cleaning and sanitization before modeling**

👉 This step defines **how the model learns and how it is evaluated**

---

### 2. Why Time-Based Splitting Is Necessary

Unlike standard machine learning tasks, financial data is **time-series data**.

### ❗ Key Constraint:
> Future data must never influence past predictions

---

### Problems with Random Splitting ❌
- Breaks temporal order  
- Causes **data leakage**  
- Leads to overly optimistic performance  

---

### Solution: Time-Based Splitting ✅

Data is split based on time:

- **Training set** → past data  
- **Validation set** → slightly more recent data  
- **Test set** → most recent data  

👉 Mimics real-world forecasting:
- Train on history  
- Validate on recent past  
- Test on unseen future  

---

### 3. Split Structure

The dataset is divided into three sequential periods:

| Split | Time Range | Purpose |
|------|-----------|--------|
| Train | First 6 months | Model learning |
| Validation | Next 3 months | Hyperparameter tuning |
| Test | Final 3 months | Final evaluation |

---

### 4. Data Sanitization (Critical Step)

Before model training, each split is cleaned.

#### 4.1 Handle Infinite Values
- Convert ±∞ → missing values  

Why:
- Can occur due to ratios or percentage changes  
- Breaks many ML models  


#### 4.2 Outlier Clipping

Each feature is clipped to:
- 1st percentile (lower bound)  
- 99th percentile (upper bound)  

Why:
- Financial data contains extreme spikes  
- Reduces instability without removing data  


#### 4.3 Remove Remaining Missing Values

- Drop rows with any missing values  

Why:
- Ensures clean input for models  
- Keeps feature-label alignment consistent  

---

### 5. Final Output Structure

The result is a structured container with:

- `X_train`, `y_train`  
- `X_val`, `y_val`  
- `X_test`, `y_test`  

---

### 6. Why This Design Is Important

This design ensures:

#### ✔ No Data Leakage
- Future information never enters training  

#### ✔ Realistic Evaluation
- Model is tested on unseen future data  

#### ✔ Stable Training
- Outliers and invalid values are controlled  

#### ✔ Clean Separation of Roles
- Train → learning  
- Validation → tuning  
- Test → final performance  

---

### 7. Summary

This section implements a **robust, time-aware data splitting strategy** that:

- Preserves temporal order  
- Prevents leakage  
- Cleans data before modeling  
- Produces reliable evaluation results  

👉 It ensures that model performance reflects **real-world predictive ability**, not artificial accuracy

In [19]:
# Cell 3 — Train/Validation/Test split (time-based)
#
# This cell creates chronological train / validation / test splits.
# No shuffling is used because future rows must never influence earlier training rows.
# A final numeric sanitization step is also applied here before model fitting.

from dataclasses import dataclass  # lightweight container for train/validation/test partitions
from sklearn.metrics import accuracy_score, roc_auc_score  # evaluation metrics used later in the notebook


@dataclass
class Split:
    X_train: pd.DataFrame  # training features
    y_train: pd.Series  # training labels
    X_val: pd.DataFrame  # validation features
    y_val: pd.Series  # validation labels
    X_test: pd.DataFrame  # test features
    y_test: pd.Series  # test labels


def time_splits(X_with_date: pd.DataFrame, y: pd.Series, date_col: str = "date") -> Split:
    df = X_with_date.copy()  # work on a copy so the caller's DataFrame is unchanged
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")  # make sure the split column is datetime

    start = df[date_col].min()  # first available date in the feature table
    cut_train = start + pd.DateOffset(months=6)  # end of training window
    cut_val = start + pd.DateOffset(months=9)  # end of validation window / start of test window

    train_mask = df[date_col] < cut_train  # first 6 months
    val_mask = (df[date_col] >= cut_train) & (df[date_col] < cut_val)  # months 6 to 9
    test_mask = df[date_col] >= cut_val  # final 3 months

    X_train = df.loc[train_mask].drop(columns=[date_col])  # training features without the date column
    y_train = y.loc[train_mask]  # training labels aligned by the same mask

    X_val = df.loc[val_mask].drop(columns=[date_col])  # validation features without the date column
    y_val = y.loc[val_mask]  # validation labels aligned by the same mask

    X_test = df.loc[test_mask].drop(columns=[date_col])  # test features without the date column
    y_test = y.loc[test_mask]  # test labels aligned by the same mask

    def _sanitize(X_part: pd.DataFrame, y_part: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
        Xs = X_part.replace([np.inf, -np.inf], np.nan).copy()  # convert infinite values to NaN before cleanup
        for c in Xs.columns:  # process each feature column independently
            lo, hi = Xs[c].quantile(0.01), Xs[c].quantile(0.99)  # robust clipping bounds
            Xs[c] = Xs[c].clip(lo, hi)  # clip extreme outliers to stabilize later models
        keep = ~Xs.isna().any(axis=1)  # keep only rows with no remaining missing values
        Xs = Xs.loc[keep].reset_index(drop=True)  # filtered feature matrix
        ys = y_part.loc[keep].reset_index(drop=True)  # filtered label vector with the same rows kept
        return Xs, ys  # return cleaned features and labels

    X_train, y_train = _sanitize(X_train, y_train)  # sanitize the training split
    X_val, y_val = _sanitize(X_val, y_val)  # sanitize the validation split
    X_test, y_test = _sanitize(X_test, y_test)  # sanitize the test split

    return Split(X_train, y_train, X_val, y_val, X_test, y_test)  # package all splits into one dataclass


split_btc = time_splits(X_btc, y_btc)  # create BTC train/val/test partitions
split_eth = time_splits(X_eth, y_eth)  # create ETH train/val/test partitions

{
    "btc": (split_btc.X_train.shape, split_btc.X_val.shape, split_btc.X_test.shape),  # quick shape check for BTC
    "eth": (split_eth.X_train.shape, split_eth.X_val.shape, split_eth.X_test.shape),  # quick shape check for ETH
}

{'btc': ((181, 57), (92, 57), (61, 57)),
 'eth': ((181, 57), (92, 57), (61, 57))}

## Section 3.2 - Model Selection & Validation Utilities

### 1. What This Section Does

This section implements the **model evaluation and hyperparameter tuning layer** of the pipeline.

Its purpose is to:

- Define a **standard evaluation function** for classifiers  
- Provide a **validation-based model selection framework**  
- Identify the **best model configuration (hyperparameters)**  
- Refit the selected model using all available pre-test data  

👉 This is where we decide **which model works best and how to configure it**

---

### 2. Why This Step Is Necessary

After feature engineering and data splitting, we still face a critical problem:

> Which model configuration should we use?


#### ❗ Without Proper Model Selection

- Models may **overfit** the training data  
- Performance may not generalize to unseen data  
- Results may depend on arbitrary parameter choices  


#### ✅ Solution: Validation-Based Selection

We use:

- **Training set** → fit models  
- **Validation set** → compare models  

👉 Ensures model selection is based on **out-of-sample performance**

---

### 3. Evaluation Metrics

Two key metrics are used:

#### **🎯 Accuracy**

- Measures the fraction of correct predictions  
- Simple and intuitive  

Limitation:
- Accuracy has limitations when samples are imbalanced.

#### **📈 ROC-AUC**

- Measures how well the model ranks positive vs negative cases  
- Threshold-independent  

Advantage:
- More robust in classification problems  

---

### 4. Hyperparameter Tuning Strategy

This section uses a **grid search on the validation set**.

#### 4.1 What Is Being Tuned

Each model is defined by:
- A function that builds the model  
- A parameter grid (possible configurations)  

Example:
- regularization strength  
- tree depth  
- learning rate  

#### 4.2 How Grid Search Works

The process:

1. Generate all parameter combinations  
2. For each combination:
   - Train on **training set**  
   - Evaluate on **validation set**  
3. Track the best-performing configuration   

---

### 5. Summary

This section builds a **systematic and reliable model selection framework** that:

- Evaluates models using consistent metrics  
- Searches for optimal hyperparameters  
- Avoids data leakage  
- Produces a well-trained model ready for final testing  

👉 It ensures that the chosen model is **not just trained, but properly validated and optimized**

In [20]:
# Cell 4 — Model selection utilities
#
# This cell contains reusable evaluation and tuning helpers shared by all models below.
# `evaluate_classifier(...)` calculates Accuracy and ROC-AUC.
# `tune_on_validation(...)` searches a parameter grid on the validation set, then refits the best model on train+val.

from itertools import product  # Cartesian product for grid-search parameter combinations
import time  # elapsed-time reporting during tuning
from math import prod  # multiply parameter-grid lengths to count total combinations
from sklearn.pipeline import Pipeline  # used to inspect wrapped estimators safely


def evaluate_classifier(model, X, y) -> dict:
    """Return Accuracy + ROC-AUC.

    Implementation note:
    - We prefer `decision_function` when available because some models' `predict_proba`
      can be numerically unstable on this dataset.
    - If using `predict_proba`, we locate the probability column for class `1` via `classes_`.
    """

    pred = model.predict(X)  # hard class predictions used for Accuracy
    acc = accuracy_score(y, pred)  # fraction of correct up/down predictions

    if hasattr(model, "decision_function"):  # preferred continuous score when available
        score = model.decision_function(X)  # raw margin / score for ROC-AUC
    elif hasattr(model, "predict_proba"):  # otherwise fall back to class probabilities
        proba = model.predict_proba(X)  # predicted class probabilities
        base = model  # default to the model itself
        if isinstance(model, Pipeline):  # if wrapped in a pipeline
            base = model.named_steps.get("clf", model)  # extract the final classifier step
        classes = list(getattr(base, "classes_", [0, 1]))  # read class ordering safely
        pos_idx = classes.index(1) if 1 in classes else -1  # locate the column corresponding to class 1
        score = proba[:, pos_idx]  # use probability of class 1 as the ranking score
    else:
        score = pred  # last-resort fallback if neither scores nor probabilities are available

    try:
        auc = roc_auc_score(y, score)  # compute ROC-AUC from the continuous score
    except Exception:
        auc = np.nan  # return NaN instead of failing if ROC-AUC cannot be computed

    return {"accuracy": acc, "roc_auc": auc}  # unified metric dictionary used everywhere below


def tune_on_validation(
    build_model_fn,
    param_grid: dict[str, list],
    split: Split,
    maximize: str = "accuracy",
    verbose: bool = False,
    print_every: int = 25,
):
    # Track the best validation metrics and corresponding parameter set.
    best = None
    best_params = None

    # Expand the parameter grid into an explicit list of combinations.
    keys = list(param_grid.keys())
    total = prod([len(param_grid[k]) for k in keys]) if keys else 0
    t0 = time.perf_counter()

    # Try each configuration on the training set, then score it on the validation set only.
    for i, values in enumerate(product(*[param_grid[k] for k in keys]), start=1):
        params = dict(zip(keys, values))
        model = build_model_fn(**params)
        model.fit(split.X_train, split.y_train)
        metrics = evaluate_classifier(model, split.X_val, split.y_val)

        if best is None:
            best = metrics
            best_params = params
        else:
            # Select best by requested validation metric; break ties with validation ROC-AUC.
            if metrics[maximize] > best[maximize] or (
                metrics[maximize] == best[maximize] and metrics["roc_auc"] > best["roc_auc"]
            ):
                best = metrics
                best_params = params

        if verbose and (i % print_every == 0 or i == 1 or i == total):
            dt = time.perf_counter() - t0
            print(
                f"  tried {i}/{total} | best val_acc={best['accuracy']:.4f}, val_auc={best['roc_auc']:.4f} | elapsed={dt:.1f}s"
            )

    # Refit the selected model on train+validation so it uses all available pre-test data.
    X_tv = pd.concat([split.X_train, split.X_val], axis=0)
    y_tv = pd.concat([split.y_train, split.y_val], axis=0)
    best_model = build_model_fn(**best_params)
    best_model.fit(X_tv, y_tv)

    # Return only validation-selected information; do not evaluate on test here.
    return best_model, best_params, best


def results_row(asset: str, model_name: str, best_params: dict, val: dict) -> dict:
    # Store selection-stage metrics only (validation metrics), keeping test for Section 6.
    return {
        "asset": asset,
        "model": model_name,
        "val_accuracy": val["accuracy"],
        "val_roc_auc": val["roc_auc"],
        "best_params": best_params,
    }


all_results: list[dict] = []
trained_models: dict[tuple[str, str], object] = {}

## Section 4 — Candidate Models Training

### 1. What This Section Does

This section implements the **model training layer** of the pipeline.

Its purpose is to:

- Train multiple machine learning models on the same dataset  
- Tune their hyperparameters using the validation set  
- Compare their performance  
- Store the best model for each method  

👉 This is where we test **different modeling approaches** on the same feature set

---

### 2. Candidate Models

Different models capture different patterns:

| Model Type | Strength |
|-----------|--------|
| Linear (Logistic) | Simple, interpretable |
| Kernel (SVM) | Nonlinear boundaries |
| Tree (Decision Tree) | Rule-based, interpretable |
| Ensemble (Random Forest) | Robust, low overfitting |
| Boosting (XGBoost) | Strong predictive power |
| Neural Network (MLP) | Flexible nonlinear learning |

---

### 3. Summary

This section implements a **comprehensive model training and comparison framework** that:

- Trains multiple model types  
- Tunes hyperparameters systematically  
- Evaluates performance consistently  
- Stores the best models for later use  

👉 It identifies which modeling approach best captures the **complex dynamics of crypto price movements**

In [21]:
# Cell 5 — Logistic Regression
#
# What it is:
# - A linear classifier; it learns weights for each feature.
#
# Why standardize:
# - Logistic Regression is sensitive to feature scale, so we use `StandardScaler()`.
#
# What we tune:
# - `C`: inverse regularization strength (larger C => weaker regularization)
# - `penalty`: L1 or L2 regularization
# - `class_weight`: handle class imbalance

# 1) Logistic Regression (standardized)

from sklearn.pipeline import Pipeline  # chain preprocessing and model into one estimator
from sklearn.preprocessing import RobustScaler  # robust scaling is less sensitive to outliers than standard scaling
from sklearn.impute import SimpleImputer  # fill any remaining missing values before modeling
from sklearn.linear_model import LogisticRegression  # linear classifier used as a strong baseline


def build_lr(C: float, penalty: str, class_weight: str | None):
    # Build one Logistic Regression pipeline for a single hyperparameter combination.
    # `saga` is used because it supports both L1 and L2 penalties.
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),  # replace missing values with the column median
            ("scaler", RobustScaler()),  # scale features with a median/IQR-based transform
            (
                "clf",
                LogisticRegression(
                    C=C,  # inverse regularization strength
                    penalty=penalty,  # regularization type: L1 or L2
                    solver="saga",  # solver compatible with L1/L2 and larger feature sets
                    max_iter=10000,  # allow more optimizer iterations for convergence
                    class_weight=class_weight,  # optionally rebalance classes if the label distribution is uneven
                    n_jobs=None,  # leave parallelism at sklearn default for this estimator
                ),
            ),
        ]
    )


lr_grid = {
    "C": [0.01, 0.1, 1.0, 10.0],  # try stronger to weaker regularization
    "penalty": ["l2", "l1"],  # compare ridge-style vs lasso-style regularization
    "class_weight": [None, "balanced"],  # compare unweighted vs automatic class balancing
}

lr_btc_model, lr_btc_params, lr_btc_val = tune_on_validation(build_lr, lr_grid, split_btc)  # tune Logistic Regression on BTC using train/validation only
all_results.append(results_row("btc", "LogisticRegression", lr_btc_params, lr_btc_val))  # record BTC validation-stage Logistic Regression metrics
trained_models[("btc", "LogisticRegression")] = lr_btc_model  # store the selected/refit BTC Logistic Regression model

lr_eth_model, lr_eth_params, lr_eth_val = tune_on_validation(build_lr, lr_grid, split_eth)  # tune Logistic Regression on ETH using train/validation only
all_results.append(results_row("eth", "LogisticRegression", lr_eth_params, lr_eth_val))  # record ETH validation-stage Logistic Regression metrics
trained_models[("eth", "LogisticRegression")] = lr_eth_model  # store the selected/refit ETH Logistic Regression model

pd.DataFrame(all_results).query("model=='LogisticRegression'")  # display only Logistic Regression rows from the cumulative results table

,asset,model,val_accuracy,val_roc_auc,best_params
0,btc,LogisticRegression,0.554348,0.529663,"{'C': 10.0, 'penalty': 'l1', 'class_weight': '..."
1,eth,LogisticRegression,0.554348,0.535088,"{'C': 0.1, 'penalty': 'l1', 'class_weight': None}"


In [22]:
# Cell 6 — Support Vector Machine (SVM)
#
# What it is:
# - Finds a separating boundary (hyperplane). With an RBF kernel it can model non-linear patterns.
#
# Why standardize:
# - SVM distance computations depend on feature scale.
#
# What we tune:
# - `C`: regularization (higher C fits training data more closely)
# - `kernel`: linear vs RBF
# - `gamma`: RBF kernel width (we keep to scale/auto here)

# 2) Support Vector Machine (standardized)

from sklearn.svm import SVC  # support vector classifier
from sklearn.impute import SimpleImputer  # fill missing values before scaling/modeling
from sklearn.preprocessing import RobustScaler  # robust scaling helps when features contain outliers


def build_svm(C: float, kernel: str, gamma: str | float, class_weight: str | None):
    # Build one SVM pipeline for a single hyperparameter combination.
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),  # replace missing values with column medians
            ("scaler", RobustScaler()),  # rescale features so SVM distances are more meaningful
            (
                "clf",
                SVC(
                    C=C,  # regularization parameter controlling margin-vs-fit tradeoff
                    kernel=kernel,  # kernel choice: linear or radial basis function
                    gamma=gamma,  # kernel coefficient for RBF; ignored by linear kernel but kept for shared interface
                    probability=True,  # enable probability estimates for downstream ROC-AUC / forecast logic
                    class_weight=class_weight,  # optionally rebalance class importance
                    random_state=42,  # fixed seed for reproducibility where applicable
                ),
            ),
        ]
    )


svm_grid = {
    "C": [0.1, 1.0, 10.0],  # compare stronger vs weaker regularization
    "kernel": ["rbf", "linear"],  # compare nonlinear and linear decision boundaries
    "gamma": ["scale", "auto"],  # compare sklearn's two built-in gamma settings
    "class_weight": [None, "balanced"],  # compare unweighted vs class-balanced fitting
}

svm_btc_model, svm_btc_params, svm_btc_val = tune_on_validation(build_svm, svm_grid, split_btc)  # tune SVM on BTC using train/validation only
all_results.append(results_row("btc", "SVM", svm_btc_params, svm_btc_val))  # record BTC validation-stage SVM metrics
trained_models[("btc", "SVM")] = svm_btc_model  # store the selected/refit BTC SVM model

svm_eth_model, svm_eth_params, svm_eth_val = tune_on_validation(build_svm, svm_grid, split_eth)  # tune SVM on ETH using train/validation only
all_results.append(results_row("eth", "SVM", svm_eth_params, svm_eth_val))  # record ETH validation-stage SVM metrics
trained_models[("eth", "SVM")] = svm_eth_model  # store the selected/refit ETH SVM model

pd.DataFrame(all_results).query("model=='SVM'")  # display only SVM rows from the cumulative results table

,asset,model,val_accuracy,val_roc_auc,best_params
2,btc,SVM,0.543478,0.554343,"{'C': 10.0, 'kernel': 'linear', 'gamma': 'scal..."
3,eth,SVM,0.521739,0.435185,"{'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale', ..."


In [23]:
# Cell 7 — Decision Tree
#
# What it is:
# - A tree of if/else rules that splits the feature space.
#
# Pros/cons:
# - Interpretable, handles non-linearities
# - Can overfit if not constrained
#
# What we tune:
# - `max_depth`, `min_samples_split`, `min_samples_leaf`, `class_weight`

# 3) Decision Tree

from sklearn.tree import DecisionTreeClassifier  # single-tree classifier based on recursive feature splits


def build_dt(max_depth: int | None, min_samples_split: int, min_samples_leaf: int, class_weight: str | None):
    # Build one Decision Tree classifier for a single hyperparameter combination.
    return DecisionTreeClassifier(
        max_depth=max_depth,  # maximum tree depth; `None` means fully expand until stopping rules are hit
        min_samples_split=min_samples_split,  # minimum samples needed to split an internal node
        min_samples_leaf=min_samples_leaf,  # minimum samples required in each leaf node
        class_weight=class_weight,  # optionally rebalance class importance
        random_state=42,  # fixed seed for reproducibility
    )


dt_grid = {
    "max_depth": [None, 3, 5, 8, 12],  # compare shallow vs deeper trees
    "min_samples_split": [2, 5, 10, 20],  # constrain how easily the tree can keep splitting
    "min_samples_leaf": [1, 2, 5, 10],  # constrain how small terminal leaves are allowed to become
    "class_weight": [None, "balanced"],  # compare unweighted vs class-balanced fitting
}

dt_btc_model, dt_btc_params, dt_btc_val = tune_on_validation(build_dt, dt_grid, split_btc)  # tune Decision Tree on BTC using train/validation only
all_results.append(results_row("btc", "DecisionTree", dt_btc_params, dt_btc_val))  # record BTC validation-stage Decision Tree metrics
trained_models[("btc", "DecisionTree")] = dt_btc_model  # store the selected/refit BTC Decision Tree model

dt_eth_model, dt_eth_params, dt_eth_val = tune_on_validation(build_dt, dt_grid, split_eth)  # tune Decision Tree on ETH using train/validation only
all_results.append(results_row("eth", "DecisionTree", dt_eth_params, dt_eth_val))  # record ETH validation-stage Decision Tree metrics
trained_models[("eth", "DecisionTree")] = dt_eth_model  # store the selected/refit ETH Decision Tree model

pd.DataFrame(all_results).query("model=='DecisionTree'")  # display only Decision Tree rows from the cumulative results table

,asset,model,val_accuracy,val_roc_auc,best_params
4,btc,DecisionTree,0.554348,0.553393,"{'max_depth': 5, 'min_samples_split': 2, 'min_..."
5,eth,DecisionTree,0.619565,0.586745,"{'max_depth': None, 'min_samples_split': 5, 'm..."


In [24]:
# Cell 8 — Random Forest
#
# What it is:
# - An ensemble of many decision trees; predictions are averaged/voted.
#
# Why it often works well:
# - Reduces overfitting vs a single tree and captures non-linear feature interactions.
#
# What we tune:
# - number of trees, depth, leaf size, feature sampling

# 4) Random Forest

from sklearn.ensemble import RandomForestClassifier  # ensemble of many decision trees trained on random subsets


def build_rf(n_estimators: int, max_depth: int | None, min_samples_leaf: int, max_features: str | float, class_weight: str | None):
    # Build one Random Forest classifier for a single hyperparameter combination.
    return RandomForestClassifier(
        n_estimators=n_estimators,  # number of trees in the ensemble
        max_depth=max_depth,  # optional maximum depth for each tree
        min_samples_leaf=min_samples_leaf,  # minimum number of samples required in each leaf
        max_features=max_features,  # number/fraction of features considered at each split
        class_weight=class_weight,  # optionally rebalance class importance
        random_state=42,  # fixed seed for reproducibility
        n_jobs=-1,  # use all available CPU cores for faster fitting
    )


rf_grid = {
    "n_estimators": [200, 500],  # compare moderately sized vs larger forests
    "max_depth": [None, 5, 10, 20],  # compare unconstrained vs progressively constrained trees
    "min_samples_leaf": [1, 2, 5],  # compare more flexible vs smoother leaves
    "max_features": ["sqrt", 0.5],  # compare standard sqrt sampling vs half-feature sampling
    "class_weight": [None, "balanced"],  # compare unweighted vs class-balanced fitting
}

rf_btc_model, rf_btc_params, rf_btc_val = tune_on_validation(build_rf, rf_grid, split_btc)  # tune Random Forest on BTC using train/validation only
all_results.append(results_row("btc", "RandomForest", rf_btc_params, rf_btc_val))  # record BTC validation-stage Random Forest metrics
trained_models[("btc", "RandomForest")] = rf_btc_model  # store the selected/refit BTC Random Forest model

rf_eth_model, rf_eth_params, rf_eth_val = tune_on_validation(build_rf, rf_grid, split_eth)  # tune Random Forest on ETH using train/validation only
all_results.append(results_row("eth", "RandomForest", rf_eth_params, rf_eth_val))  # record ETH validation-stage Random Forest metrics
trained_models[("eth", "RandomForest")] = rf_eth_model  # store the selected/refit ETH Random Forest model

pd.DataFrame(all_results).query("model=='RandomForest'")  # display only Random Forest rows from the cumulative results table

,asset,model,val_accuracy,val_roc_auc,best_params
6,btc,RandomForest,0.619565,0.608448,"{'n_estimators': 200, 'max_depth': 5, 'min_sam..."
7,eth,RandomForest,0.586957,0.550682,"{'n_estimators': 500, 'max_depth': 10, 'min_sa..."


In [25]:
# Cell 9 — XGBoost
#
# What it is:
# - Gradient-boosted decision trees (strong baseline for tabular ML).
#
# Why it may take time:
# - We run a grid search; each combination trains hundreds of trees.
# - The progress prints show how many combinations have been tried.

# 5) XGBoost (if available)

xgb_available = False  # default assumption: XGBoost may not be installed in the environment
try:
    import xgboost  # import the package itself so its version can be shown
    from xgboost import XGBClassifier  # import the sklearn-compatible classifier API

    xgb_available = True  # mark XGBoost as available if import succeeds
    print("XGBoost available:", True, "| version:", xgboost.__version__)  # confirm availability and package version
except Exception as e:
    XGBClassifier = None  # define a placeholder so the name still exists if import fails
    print("XGBoost available:", False)  # report that XGBoost is unavailable
    print("Import error:", e)  # show the import exception for debugging


if xgb_available:

    def build_xgb(
        n_estimators: int,
        max_depth: int,
        learning_rate: float,
        subsample: float,
        colsample_bytree: float,
        gamma: float,
        min_child_weight: float,
        reg_lambda: float,
    ):
        # Build one XGBoost classifier for a single hyperparameter combination.
        return XGBClassifier(
            n_estimators=n_estimators,  # number of boosting rounds / trees
            max_depth=max_depth,  # maximum depth of each tree
            learning_rate=learning_rate,  # shrinkage factor applied to each boosting step
            subsample=subsample,  # fraction of rows sampled for each tree
            colsample_bytree=colsample_bytree,  # fraction of columns sampled for each tree
            gamma=gamma,  # minimum loss reduction required to split
            min_child_weight=min_child_weight,  # minimum sum of Hessian / child weight needed in a leaf
            reg_lambda=reg_lambda,  # L2 regularization on leaf weights
            objective="binary:logistic",  # binary classification objective with logistic output
            eval_metric="logloss",  # evaluation metric used internally during fitting
            random_state=42,  # fixed seed for reproducibility
            n_jobs=-1,  # use all available CPU cores
            tree_method="hist",  # histogram tree builder for faster tabular training
        )

    USE_FAST_GRID = True  # use the smaller grid by default because the full search space is very large

    if USE_FAST_GRID:
        xgb_grid = {
            "n_estimators": [400, 800],  # compare moderately large vs larger boosting rounds
            "max_depth": [4, 6, 8],  # compare medium vs deeper trees
            "learning_rate": [0.03, 0.05, 0.1],  # compare slower vs faster boosting updates
            "subsample": [0.6],  # compare stronger vs lighter row subsampling, [0.6, 0.8, 0.9]
            "colsample_bytree": [0.6],  # compare stronger vs lighter column subsampling, [0.6, 0.8, 0.9]
            "gamma": [0.0],  # compare no split penalty vs moderate split penalty, [0.0, 0.5]
            "min_child_weight": [1.0],  # compare more flexible vs more conservative leaf growth, [1.0, 3.0]
            "reg_lambda": [1.0],  # compare lighter vs stronger L2 regularization, [1.0, 5.0]
        }
    else:
        xgb_grid = {
            "n_estimators": [300, 600, 900],  # larger alternative search space for more exhaustive tuning
            "max_depth": [4, 6, 8],
            "learning_rate": [0.01, 0.03, 0.05, 0.1],
            "subsample": [0.6, 0.75, 0.9],
            "colsample_bytree": [0.6, 0.75, 0.9],
            "gamma": [0.0, 0.5, 1.0],
            "min_child_weight": [1.0, 3.0, 5.0],
            "reg_lambda": [1.0, 5.0, 10.0],
        }

    print("Tuning XGBoost for BTC...")  # progress message before the BTC search starts
    xgb_btc_model, xgb_btc_params, xgb_btc_val = tune_on_validation(
        build_xgb,  # model-builder function
        xgb_grid,  # XGBoost hyperparameter grid
        split_btc,  # BTC split object (selection uses train/validation only)
        verbose=True,  # print progress during the search
        print_every=20,  # print after every 20 parameter combinations
    )
    all_results.append(results_row("btc", "XGBoost", xgb_btc_params, xgb_btc_val))  # record BTC validation-stage XGBoost metrics
    trained_models[("btc", "XGBoost")] = xgb_btc_model  # store the selected/refit BTC XGBoost model

    print("Tuning XGBoost for ETH...")  # progress message before the ETH search starts
    xgb_eth_model, xgb_eth_params, xgb_eth_val = tune_on_validation(
        build_xgb,  # model-builder function
        xgb_grid,  # XGBoost hyperparameter grid
        split_eth,  # ETH split object (selection uses train/validation only)
        verbose=True,  # print progress during the search
        print_every=20,  # print after every 20 parameter combinations
    )
    all_results.append(results_row("eth", "XGBoost", xgb_eth_params, xgb_eth_val))  # record ETH validation-stage XGBoost metrics
    trained_models[("eth", "XGBoost")] = xgb_eth_model  # store the selected/refit ETH XGBoost model

    pd.DataFrame(all_results).query("model=='XGBoost'")  # display only XGBoost rows from the cumulative results table

XGBoost available: True | version: 3.2.0
Tuning XGBoost for BTC...
  tried 1/18 | best val_acc=0.5870, val_auc=0.5956 | elapsed=0.6s
  tried 18/18 | best val_acc=0.6196, val_auc=0.6312 | elapsed=12.8s
Tuning XGBoost for ETH...
  tried 1/18 | best val_acc=0.4891, val_auc=0.5458 | elapsed=0.5s
  tried 18/18 | best val_acc=0.5326, val_auc=0.5658 | elapsed=12.9s


In [26]:
# Cell 10 — Artificial Neural Network (MLP)
#
# What it is:
# - A small feed-forward neural network (Multi-Layer Perceptron) for tabular classification.
#
# Why standardize:
# - Neural nets train much more stably when features are scaled.
#
# What we tune:
# - network size (hidden layers), L2 regularization (alpha), learning rate

# 6) Artificial Neural Network (MLP, standardized)

from sklearn.neural_network import MLPClassifier  # feed-forward neural network classifier
from sklearn.impute import SimpleImputer  # fill missing values before scaling/modeling
from sklearn.preprocessing import RobustScaler  # robust scaling improves neural-network training stability on outlier-heavy data


def build_mlp(hidden_layer_sizes: tuple[int, ...], alpha: float, learning_rate_init: float):
    # Build one MLP pipeline for a single hyperparameter combination.
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),  # replace missing values with the column median
            ("scaler", RobustScaler()),  # scale features before neural-network optimization
            (
                "clf",
                MLPClassifier(
                    hidden_layer_sizes=hidden_layer_sizes,  # hidden-layer architecture
                    alpha=alpha,  # L2 regularization strength
                    learning_rate_init=learning_rate_init,  # initial learning rate for gradient-based optimization
                    max_iter=3000,  # allow more epochs for convergence
                    early_stopping=True,  # stop training when validation score no longer improves
                    n_iter_no_change=20,  # patience for early stopping
                    random_state=42,  # fixed seed for reproducibility
                ),
            ),
        ]
    )


mlp_grid = {
    "hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],  # compare smaller vs larger network structures
    "alpha": [1e-4, 1e-3, 1e-2],  # compare lighter vs stronger regularization
    "learning_rate_init": [1e-3, 5e-4],  # compare faster vs slower initial learning rates
}

mlp_btc_model, mlp_btc_params, mlp_btc_val = tune_on_validation(build_mlp, mlp_grid, split_btc)  # tune MLP on BTC using train/validation only
all_results.append(results_row("btc", "ANN_MLP", mlp_btc_params, mlp_btc_val))  # record BTC validation-stage MLP metrics
trained_models[("btc", "ANN_MLP")] = mlp_btc_model  # store the selected/refit BTC MLP model

mlp_eth_model, mlp_eth_params, mlp_eth_val = tune_on_validation(build_mlp, mlp_grid, split_eth)  # tune MLP on ETH using train/validation only
all_results.append(results_row("eth", "ANN_MLP", mlp_eth_params, mlp_eth_val))  # record ETH validation-stage MLP metrics
trained_models[("eth", "ANN_MLP")] = mlp_eth_model  # store the selected/refit ETH MLP model

pd.DataFrame(all_results).query("model=='ANN_MLP'")  # display only MLP rows from the cumulative results table

,asset,model,val_accuracy,val_roc_auc,best_params
10,btc,ANN_MLP,0.521739,0.517798,"{'hidden_layer_sizes': (32,), 'alpha': 0.0001,..."
11,eth,ANN_MLP,0.500000,0.473197,"{'hidden_layer_sizes': (64,), 'alpha': 0.0001,..."


## Section 5 — Model Comparison & Feature Importance

### 1. What This Section Does

This section performs two key tasks:

1. **Compare all trained models** using validation performance  
2. **Explain the best model** by identifying the most important features  

👉 This is where we answer:
- *Which model is best?*  
- *Why is it making its predictions?*  

---

### 2. Best Model Selection

For each asset:

- Select the top-ranked model  

This produces:

- Best model for BTC  
- Best model for ETH  

---

### 3. Feature Importance Analysis

#### 🎯 Goal

To understand:
> Which features drive the model's predictions

#### Why Feature Importance Matters

- Improves interpretability  
- Provides economic / financial insights  

---

### 4. Summary

This section provides both:

#### 📊 Model-Level Insight
- Which model performs best  
- Based on validation metrics  

#### 🧠 Feature-Level Insight
- Which variables drive predictions   

👉 Together, they ensure the pipeline is not only **accurate**, but also **interpretable and trustworthy**

In [27]:
# Cell 11 — Compare model performance
#
# Goal:
# - Put every model's validation metrics into one table for BTC and ETH.
# - Sort by validation performance to pick the best model for later test simulation.

# Compare model performance (validation stage only)

results_df = pd.DataFrame(all_results)  # convert the accumulated selection-stage result dictionaries into one table
results_df = results_df.sort_values(["asset", "val_accuracy", "val_roc_auc"], ascending=[True, False, False])  # sort by asset, then by validation performance
results_df  # display the ordered validation summary table used for model selection

,asset,model,val_accuracy,val_roc_auc,best_params
8,btc,XGBoost,0.619565,0.631229,"{'n_estimators': 400, 'max_depth': 6, 'learnin..."
6,btc,RandomForest,0.619565,0.608448,"{'n_estimators': 200, 'max_depth': 5, 'min_sam..."
4,btc,DecisionTree,0.554348,0.553393,"{'max_depth': 5, 'min_samples_split': 2, 'min_..."
0,btc,LogisticRegression,0.554348,0.529663,"{'C': 10.0, 'penalty': 'l1', 'class_weight': '..."
2,btc,SVM,0.543478,0.554343,"{'C': 10.0, 'kernel': 'linear', 'gamma': 'scal..."
10,btc,ANN_MLP,0.521739,0.517798,"{'hidden_layer_sizes': (32,), 'alpha': 0.0001,..."
5,eth,DecisionTree,0.619565,0.586745,"{'max_depth': None, 'min_samples_split': 5, 'm..."
7,eth,RandomForest,0.586957,0.550682,"{'n_estimators': 500, 'max_depth': 10, 'min_sa..."
1,eth,LogisticRegression,0.554348,0.535088,"{'C': 0.1, 'penalty': 'l1', 'class_weight': None}"
9,eth,XGBoost,0.532609,0.565789,"{'n_estimators': 400, 'max_depth': 6, 'learnin..."


In [28]:
# Cell 12 — Feature importance
#
# Goal:
# - Explain which features matter most for the best model (for interpretability).
#
# Methods used:
# - Tree models: use built-in `feature_importances_`
# - Linear models: use absolute value of coefficients `|coef|`
# - Other models: use permutation importance (how much accuracy drops when a feature is shuffled)

# Feature importance
# - For tree-based models: native feature_importances_
# - For linear models: |coef|
# - Otherwise: permutation importance on the test set

from sklearn.inspection import permutation_importance


def get_feature_importance(model, X: pd.DataFrame, y: pd.Series, feature_names: list[str], top_k: int = 20) -> pd.DataFrame:
    # If the model is wrapped in a Pipeline, extract the final estimator first.
    clf = model
    if isinstance(model, Pipeline):
        clf = model.named_steps.get("clf", model)

    # Tree models expose impurity-based feature importance directly.
    if hasattr(clf, "feature_importances_"):
        imp = pd.Series(clf.feature_importances_, index=feature_names)
        out = imp.sort_values(ascending=False).head(top_k).reset_index()
        out.columns = ["feature", "importance"]
        out["method"] = "native_feature_importances_"
        return out

    # Linear models expose coefficients; absolute value gives a simple importance ranking.
    if hasattr(clf, "coef_"):
        coefs = np.ravel(clf.coef_)
        imp = pd.Series(np.abs(coefs), index=feature_names)
        out = imp.sort_values(ascending=False).head(top_k).reset_index()
        out.columns = ["feature", "importance"]
        out["method"] = "abs_coef_"
        return out

    # For models without native importance, estimate importance by shuffling each feature.
    perm = permutation_importance(model, X, y, n_repeats=10, random_state=42, scoring="accuracy")
    imp = pd.Series(perm.importances_mean, index=feature_names)
    out = imp.sort_values(ascending=False).head(top_k).reset_index()
    out.columns = ["feature", "importance"]
    out["method"] = "permutation_accuracy"
    return out


def best_model_for_asset(asset: str) -> tuple[str, object]:
    # best by validation accuracy then roc_auc
    sub = results_df[results_df["asset"] == asset].copy()
    sub = sub.sort_values(["val_accuracy", "val_roc_auc"], ascending=[False, False])
    best_row = sub.iloc[0]
    model_name = best_row["model"]
    return model_name, trained_models[(asset, model_name)]


# BTC best
btc_best_name, btc_best_model = best_model_for_asset("btc")
btc_imp = get_feature_importance(btc_best_model, split_btc.X_test, split_btc.y_test, list(split_btc.X_test.columns), top_k=25)

# ETH best
eth_best_name, eth_best_model = best_model_for_asset("eth")
eth_imp = get_feature_importance(eth_best_model, split_eth.X_test, split_eth.y_test, list(split_eth.X_test.columns), top_k=25)

btc_best_name, btc_imp, eth_best_name, eth_imp

('XGBoost',
                    feature  importance                       method
 0               qqq_ret_1d    0.030395  native_feature_importances_
 1           btc_ret1_x_vix    0.024750  native_feature_importances_
 2               btc_rsi_14    0.022892  native_feature_importances_
 3                 gt_pca_1    0.022507  native_feature_importances_
 4          btc_macd_signal    0.022143  native_feature_importances_
 5             btc_RSI_lag1    0.021353  native_feature_importances_
 6   btc_volume_change_lag1    0.021121  native_feature_importances_
 7               qqq_Volume    0.020999  native_feature_importances_
 8                gt_btc_pm    0.020508  native_feature_importances_
 9    btc_vol_chg_1d_x_bull    0.020229  native_feature_importances_
 10            btc_vol_lag1    0.020015  native_feature_importances_
 11               gt_crypto    0.019820  native_feature_importances_
 12          btc_vol_chg_1d    0.019756  native_feature_importances_
 13           btc_rsi_

## Section 6 — Final Test-Set Evaluation

### 1. What This Section Does

This section performs the **final evaluation of the entire pipeline**.

Its purpose is to:

- Select the **best model for each asset (BTC / ETH)** based on validation performance  
- Apply the selected model to the **held-out test set**  
- Report final performance metrics:
  - Test Accuracy  
  - Test ROC-AUC  

👉 This is the **true performance measurement** of the system

---

### 2. Why This Step Is Critical

All previous steps (feature engineering, tuning, model selection) are **preparation**.

This step answers the key question:

> How well does the model perform in the “real world”?

#### ❗ Important Principle

The test set:
- Has never been used for training  
- Has never been used for model selection  

👉 Ensures **unbiased evaluation**

---

## 3. Summary

This section completes the pipeline by:

- Selecting the best model using validation performance  
- Evaluating it on unseen test data  
- Reporting final performance metrics  

👉 It provides the **most reliable estimate** of how well the model will perform in real-world crypto prediction tasks

In [29]:
# Cell 13 — Final test-set performance (best validation ROC-AUC model per asset)
#
# This cell performs the final simulation exactly as requested:
# 1) For each asset (BTC/ETH), select the model with the highest validation ROC-AUC.
# 2) Use that selected model to predict on the held-out test set.
# 3) Output only the test Accuracy and test ROC-AUC.


def select_best_model_name_by_val_roc_auc(asset: str) -> str:
    # Keep only rows for the requested asset.
    sub = results_df[results_df["asset"] == asset].copy()
    # Rank by validation ROC-AUC first (primary), then validation accuracy as tie-breaker.
    sub = sub.sort_values(["val_roc_auc", "val_accuracy"], ascending=[False, False])
    # Return the top model name under this ranking rule.
    return str(sub.iloc[0]["model"])


def positive_class_score(model, X: pd.DataFrame) -> np.ndarray:
    # Build continuous scores for ROC-AUC computation on test data.
    if hasattr(model, "decision_function"):
        # Convert decision scores to a probability-like range for consistency.
        raw = model.decision_function(X)
        return 1.0 / (1.0 + np.exp(-np.asarray(raw, dtype=float)))

    if hasattr(model, "predict_proba"):
        # Use class-1 probability directly if available.
        proba = model.predict_proba(X)
        base = model
        if isinstance(model, Pipeline):
            # Read class ordering from the final classifier when model is a pipeline.
            base = model.named_steps.get("clf", model)
        classes = list(getattr(base, "classes_", [0, 1]))
        pos_idx = classes.index(1) if 1 in classes else -1
        return np.asarray(proba[:, pos_idx], dtype=float)

    # Fallback to hard predictions if neither score interface exists.
    return np.asarray(model.predict(X), dtype=float)


def evaluate_selected_model_on_test(asset: str, split: Split) -> dict:
    # Select the best model using validation ROC-AUC.
    best_name = select_best_model_name_by_val_roc_auc(asset)
    # Retrieve that selected model object.
    model = trained_models[(asset, best_name)]

    # Generate hard predictions for test Accuracy.
    y_pred = model.predict(split.X_test)
    # Generate continuous scores for test ROC-AUC.
    y_score = positive_class_score(model, split.X_test)

    # Compute held-out test Accuracy.
    test_acc = accuracy_score(split.y_test, y_pred)
    # Compute held-out test ROC-AUC.
    try:
        test_auc = roc_auc_score(split.y_test, y_score)
    except Exception:
        test_auc = np.nan

    # Return the requested final metrics for one asset.
    return {
        "asset": asset,
        "selected_model_by_val_roc_auc": best_name,
        "test_accuracy": float(test_acc),
        "test_roc_auc": float(test_auc) if np.isfinite(test_auc) else np.nan,
    }


# Evaluate BTC using the model selected by best validation ROC-AUC.
btc_test_result = evaluate_selected_model_on_test("btc", split_btc)
# Evaluate ETH using the model selected by best validation ROC-AUC.
eth_test_result = evaluate_selected_model_on_test("eth", split_eth)

# Combine BTC and ETH results into one final summary table.
final_test_results = pd.DataFrame([btc_test_result, eth_test_result]).copy()

# Final required output: test Accuracy and test ROC-AUC for BTC and ETH.
final_test_results

,asset,selected_model_by_val_roc_auc,test_accuracy,test_roc_auc
0,btc,XGBoost,0.540984,0.571121
1,eth,DecisionTree,0.524590,0.486472
